# No Man's Sky.

https://store.steampowered.com/app/275850/No_Mans_Sky/

### Background

After nearly a decade, <b>No Man's Sky</b> has garnered about 224k reviews on Steam. With that many reviews, can a handful of reviews generally sum up the sentiment of how the players felt throughout the years? It is absolutely possible, and it is possible through <b>Natural Language Processing</b>.

<b>Natural Language Processing</b> is a concept in artificial intelligence that uses machine learning to assist computers to recognise, understand, and process human language. With the amount of reviews <b>No Man's Sky</b> has accumulated over the years, going through every single review by hand would be painstakingly time-consuming. <i>Text summarisation</i>, a concept built on extracting and processing key concepts and values, can sieve through and stack rank reviews based on how similar they are to other reviews. Reviews are set up as clusters all interconnected to each other and are then ranked by the commonality of words used. Once all the clusters have been ranked, each review will have its rank attached to it and the highest ranked review can be considered as the "general consensus".

The main libraries used here will be the <i>Natural Language Toolkit</i> and <i>NetworkX</i> libraries, but other concepts such as <i>cosine similarity</i> from linear algebra and <i>graphs</i> from data structures are also important to understand.

In [1]:
import pandas as pd
import re

import math
import numpy as np
import nltk
from nltk.corpus import stopwords
from nltk.cluster.util import cosine_distance
import networkx as nx

In [2]:
nms_initial = pd.read_json("./mns-data/nms_initial.json")
nms_foundation = pd.read_json("./mns-data/nms_foundation.json")
nms_pathfinder = pd.read_json("./mns-data/nms_pathfinder.json")
nms_atlas_rises = pd.read_json("./mns-data/nms_atlas_rises.json")
nms_next = pd.read_json("./mns-data/nms_next.json")
nms_abyss = pd.read_json("./mns-data/nms_abyss.json")
nms_visions = pd.read_json("./mns-data/nms_visions.json")
nms_beyond = pd.read_json("./mns-data/nms_beyond.json")
nms_synthesis = pd.read_json("./mns-data/nms_synthesis.json")
nms_living_ship = pd.read_json("./mns-data/nms_living_ship.json")
nms_exo_mech = pd.read_json("./mns-data/nms_exo_mech.json")
nms_crossplay = pd.read_json("./mns-data/nms_crossplay.json")
nms_desolation = pd.read_json("./mns-data/nms_desolation.json")
nms_origins = pd.read_json("./mns-data/nms_origins.json")
nms_next_generation = pd.read_json("./mns-data/nms_next_generation.json")
nms_companions = pd.read_json("./mns-data/nms_companions.json")
nms_expeditions = pd.read_json("./mns-data/nms_expeditions.json")
nms_prisms = pd.read_json("./mns-data/nms_prisms.json")

Importing all the needed libraries and the 18 patches <b>No Man's Sky</b> went through in order to get to the overall positive rating status on Steam.

### Part I: Cleaning

While all the reviews pulled from Steam were in labeled as English, the reality is far more complicated than that. Some reviews, while marked as English, are actually not in English due to human error. Other reviews, while also marked as English, are either partly in English or a series of symbols like emojis or ASCII art.

There is no easy way to clean up every single review when done manually unless <i>Large Language Models</i> are involved, so using <i>regular expressions</i> is the next best thing. It is not perfect, and all it does is filter out some reviews that do not meet a certain criteria, but it is still quite helpful if a batch of reviews are mostly low-effort attempts and not useful for summarising. For a patch like the <i>Initial</i> patch with 32k reviews, that just might help with the runtime when summarising.

In [3]:
checking = re.compile(r'[a-zA-Z]')
lower_bound = 75
upper_bound = 500

#### Cleaning up: Initial

In [4]:
initial_id = []

for i in range(len(nms_initial)):
    if checking.search(nms_initial['Reviews'].values[i]) and (len(nms_initial['Reviews'].values[i]) >= lower_bound and len(nms_initial['Reviews'].values[i]) <= upper_bound):
        continue
    else:
        initial_id.append(nms_initial['id'].values[i])

for i in range(len(initial_id)):
    nms_initial.drop(nms_initial.loc[(nms_initial['id'] == initial_id[i])].index, inplace=True)

#### Cleaning up: Foundation

In [5]:
foundation_id = []

for i in range(len(nms_foundation)):
    if checking.search(nms_foundation['Reviews'].values[i]) and (len(nms_foundation['Reviews'].values[i]) >= lower_bound and len(nms_foundation['Reviews'].values[i]) <= upper_bound):
        continue
    else:
        foundation_id.append(nms_foundation['id'].values[i])

for i in range(len(foundation_id)):
    nms_foundation.drop(nms_foundation.loc[(nms_foundation['id'] == foundation_id[i])].index, inplace=True)

#### Cleaning up: Pathfinder

In [6]:
pathfinder_id = []

for i in range(len(nms_pathfinder)):
    if checking.search(nms_pathfinder['Reviews'].values[i]) and (len(nms_pathfinder['Reviews'].values[i]) >= lower_bound and len(nms_pathfinder['Reviews'].values[i]) <= upper_bound):
        continue
    else:
        pathfinder_id.append(nms_pathfinder['id'].values[i])

for i in range(len(pathfinder_id)):
    nms_pathfinder.drop(nms_pathfinder.loc[(nms_pathfinder['id'] == pathfinder_id[i])].index, inplace=True)

#### Cleaning up: Atlas Rises

In [7]:
ar_id = []

for i in range(len(nms_atlas_rises)):
    if checking.search(nms_atlas_rises['Reviews'].values[i]) and (len(nms_atlas_rises['Reviews'].values[i]) >= lower_bound and len(nms_atlas_rises['Reviews'].values[i]) <= upper_bound):
        continue
    else:
        ar_id.append(nms_atlas_rises['id'].values[i])

for i in range(len(ar_id)):
    nms_atlas_rises.drop(nms_atlas_rises.loc[(nms_atlas_rises['id'] == ar_id[i])].index, inplace=True)

#### Cleaning up: NEXT

In [8]:
next_id = []

for i in range(len(nms_next)):
    if checking.search(nms_next['Reviews'].values[i]) and (len(nms_next['Reviews'].values[i]) >= lower_bound and len(nms_next['Reviews'].values[i]) <= upper_bound):
        continue
    else:
        next_id.append(nms_next['id'].values[i])

for i in range(len(next_id)):
    nms_next.drop(nms_next.loc[(nms_next['id'] == next_id[i])].index, inplace=True)

#### Cleaning up: Abyss

In [9]:
abyss_id = []

for i in range(len(nms_abyss)):
    if checking.search(nms_abyss['Reviews'].values[i]) and (len(nms_abyss['Reviews'].values[i]) >= lower_bound and len(nms_abyss['Reviews'].values[i]) <= upper_bound):
        continue
    else:
        abyss_id.append(nms_abyss['id'].values[i])

for i in range(len(abyss_id)):
    nms_abyss.drop(nms_abyss.loc[(nms_abyss['id'] == abyss_id[i])].index, inplace=True)

#### Cleaning up: Visions

In [10]:
visions_id = []

for i in range(len(nms_visions)):
    if checking.search(nms_visions['Reviews'].values[i]) and (len(nms_visions['Reviews'].values[i]) >= lower_bound and len(nms_visions['Reviews'].values[i]) <= upper_bound):
        continue
    else:
        visions_id.append(nms_visions['id'].values[i])

for i in range(len(visions_id)):
    nms_visions.drop(nms_visions.loc[(nms_visions['id'] == visions_id[i])].index, inplace=True)

#### Cleaning up: Beyond

In [11]:
beyond_id = []

for i in range(len(nms_beyond)):
    if checking.search(nms_beyond['Reviews'].values[i]) and (len(nms_beyond['Reviews'].values[i]) >= lower_bound and len(nms_beyond['Reviews'].values[i]) <= upper_bound):
        continue
    else:
        beyond_id.append(nms_beyond['id'].values[i])

for i in range(len(beyond_id)):
    nms_beyond.drop(nms_beyond.loc[(nms_beyond['id'] == beyond_id[i])].index, inplace=True)

#### Cleaning up: Synthesis

In [12]:
synthesis_id = []

for i in range(len(nms_synthesis)):
    if checking.search(nms_synthesis['Reviews'].values[i]) and (len(nms_synthesis['Reviews'].values[i]) >= lower_bound and len(nms_synthesis['Reviews'].values[i]) <= upper_bound):
        continue
    else:
        synthesis_id.append(nms_synthesis['id'].values[i])

for i in range(len(synthesis_id)):
    nms_synthesis.drop(nms_synthesis.loc[(nms_synthesis['id'] == synthesis_id[i])].index, inplace=True)

#### Cleaning up: Living Ship

In [13]:
living_ship_id = []

for i in range(len(nms_living_ship)):
    if checking.search(nms_living_ship['Reviews'].values[i]) and (len(nms_living_ship['Reviews'].values[i]) >= lower_bound and len(nms_living_ship['Reviews'].values[i]) <= upper_bound):
        continue
    else:
        living_ship_id.append(nms_living_ship['id'].values[i])

for i in range(len(living_ship_id)):
    nms_living_ship.drop(nms_living_ship.loc[(nms_living_ship['id'] == living_ship_id[i])].index, inplace=True)

#### Cleaning up: Exo Mech

In [14]:
exo_mech_id = []

for i in range(len(nms_exo_mech)):
    if checking.search(nms_exo_mech['Reviews'].values[i]) and (len(nms_exo_mech['Reviews'].values[i]) >= lower_bound and len(nms_exo_mech['Reviews'].values[i]) <= upper_bound):
        continue
    else:
        exo_mech_id.append(nms_exo_mech['id'].values[i])

for i in range(len(exo_mech_id)):
    nms_exo_mech.drop(nms_exo_mech.loc[(nms_exo_mech['id'] == exo_mech_id[i])].index, inplace=True)

#### Cleaning up: Crossplay

In [15]:
crossplay_id = []

for i in range(len(nms_crossplay)):
    if checking.search(nms_crossplay['Reviews'].values[i]) and (len(nms_crossplay['Reviews'].values[i]) >= lower_bound and len(nms_crossplay['Reviews'].values[i]) <= upper_bound):
        continue
    else:
        crossplay_id.append(nms_crossplay['id'].values[i])

for i in range(len(crossplay_id)):
    nms_crossplay.drop(nms_crossplay.loc[(nms_crossplay['id'] == crossplay_id[i])].index, inplace=True)

#### Cleaning up: Desolation

In [16]:
desolation_id = []

for i in range(len(nms_desolation)):
    if checking.search(nms_desolation['Reviews'].values[i]) and (len(nms_desolation['Reviews'].values[i]) >= lower_bound and len(nms_desolation['Reviews'].values[i]) <= upper_bound):
        continue
    else:
        desolation_id.append(nms_desolation['id'].values[i])

for i in range(len(desolation_id)):
    nms_desolation.drop(nms_desolation.loc[(nms_desolation['id'] == desolation_id[i])].index, inplace=True)

#### Cleaning up: Origins

In [17]:
origins_id = []

for i in range(len(nms_origins)):
    if checking.search(nms_origins['Reviews'].values[i]) and (len(nms_origins['Reviews'].values[i]) >= lower_bound and len(nms_origins['Reviews'].values[i]) <= upper_bound):
        continue
    else:
        origins_id.append(nms_origins['id'].values[i])

for i in range(len(origins_id)):
    nms_origins.drop(nms_origins.loc[(nms_origins['id'] == origins_id[i])].index, inplace=True)

#### Cleaning up: Next Generation

In [18]:
next_gen_id = []

for i in range(len(nms_next_generation)):
    if checking.search(nms_next_generation['Reviews'].values[i]) and (len(nms_next_generation['Reviews'].values[i]) >= lower_bound and len(nms_next_generation['Reviews'].values[i]) <= upper_bound):
        continue
    else:
        next_gen_id.append(nms_next_generation['id'].values[i])

for i in range(len(next_gen_id)):
    nms_next_generation.drop(nms_next_generation.loc[(nms_next_generation['id'] == next_gen_id[i])].index, inplace=True)

#### Cleaning up: Companions

In [19]:
companions_id = []

for i in range(len(nms_companions)):
    if checking.search(nms_companions['Reviews'].values[i]) and (len(nms_companions['Reviews'].values[i]) >= lower_bound and len(nms_companions['Reviews'].values[i]) <= upper_bound):
        continue
    else:
        companions_id.append(nms_companions['id'].values[i])

for i in range(len(companions_id)):
    nms_companions.drop(nms_companions.loc[(nms_companions['id'] == companions_id[i])].index, inplace=True)

#### Cleaning up: Expeditions

In [20]:
expeditions_id = []

for i in range(len(nms_expeditions)):
    if checking.search(nms_expeditions['Reviews'].values[i]) and (len(nms_expeditions['Reviews'].values[i]) >= lower_bound and len(nms_expeditions['Reviews'].values[i]) <= upper_bound):
        continue
    else:
        expeditions_id.append(nms_expeditions['id'].values[i])

for i in range(len(expeditions_id)):
    nms_expeditions.drop(nms_expeditions.loc[(nms_expeditions['id'] == expeditions_id[i])].index, inplace=True)

#### Cleaning up: Prisms

In [21]:
prisms_id = []

for i in range(len(nms_prisms)):
    if checking.search(nms_prisms['Reviews'].values[i]) and (len(nms_prisms['Reviews'].values[i]) >= lower_bound and len(nms_prisms['Reviews'].values[i]) <= upper_bound):
        continue
    else:
        prisms_id.append(nms_prisms['id'].values[i])

for i in range(len(prisms_id)):
    nms_prisms.drop(nms_prisms.loc[(nms_prisms['id'] == prisms_id[i])].index, inplace=True)

There are two conditions set:
- All reviews must be between 75 and 500 characters
- All reviews also must have at least a letter (from the English alphabet) in it

These conditions remove all "unhelpful" reviews, especially if they tend to be either a single word or a series of symbols that represent some sort of emoji or ASCII art.

The upper and lower bounds are determined by the [average number of characters per word](https://www.wolframalpha.com/input/?i=average+english+word+length), the [average number of words per sentence](https://lifelong-learning.ox.ac.uk/about/sentence-length), and the [average number of sentences per paragraph](https://owl.purdue.edu/owl/general_writing/academic_writing/paragraphs_and_paragraphing/paragraphing.html). Since the average English word is about five characters, then the average sentence is about 75 to 100 characters. If a paragraph is somewhere between three to five sentences, then at the maximum a paragraph with five sentences is about 500 characters. The idea is to focus on reviews that are somewhere between a sentence to a paragraph, because generally anything more leans towards a rant, and that is especially true during the early days of release.

Another detail worth noting... Because of more "content" a lengthier review has, summarisation will more often than not default to the lengthier reviews. Cutting reviews off at paragraph length seems abstract, but it gives a healthier pool of reviews to summarise on.

### Part II: Natural language processing

In [22]:
def sentence_similarity(sent1, sent2, stopwords=None):
    if stopwords is None:
        stopwords = []
    sent1 = [w.lower() for w in sent1]
    sent2 = [w.lower() for w in sent2]
    all_words = list(set(sent1+sent2))
    
    vector1 = [0] * len(all_words)
    vector2 = [0] * len(all_words)
    
    for w in sent1:
        if w in stopwords:
            continue
        vector1[all_words.index(w)] += 1
        
    for w in sent2:
        if w in stopwords:
            continue
        vector2[all_words.index(w)] += 1
        
    return 1-cosine_distance(vector1, vector2)

def gen_sim_matrix(sentences, stop_words):
    similarity_matrix = np.zeros((len(sentences), len(sentences)))
    for idx1 in range(len(sentences)):
        for idx2 in range(len(sentences)):
            if idx1 == idx2:
                continue
            similarity_matrix[idx1][idx2] = sentence_similarity(sentences[idx1], sentences[idx2], stop_words)
            
    return similarity_matrix

def generate_summary(reviews, top_n=5):
    stop_words = stopwords.words('english')
    summarise_text = []
    sentence_similarity_matrix = gen_sim_matrix(reviews, stop_words)
    sentence_similarity_graph = nx.from_numpy_array(sentence_similarity_matrix)
    scores = nx.pagerank(sentence_similarity_graph)
    ranked_sentence = sorted(((scores[i], s) for i, s in enumerate(reviews)), reverse=True)

    for i in range(top_n):
        summarise_text.append(''.join(ranked_sentence[i][1]))
        
    return summarise_text

So! What is going on here and how does this work?

An empty correlation matrix is constructed with dimensions equal to the amount of reviews, and the matrix is filled with ranking values determined by the [cosine similarity](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.pairwise.cosine_similarity.html) between different reviews after the reviews are broken down into key values and compared with each other for commonality.

With a filled correlation matrix, a graph structure is set up using the [from_numpy_array](https://networkx.org/documentation/stable/reference/generated/networkx.convert_matrix.from_numpy_array.html) function from the <i>NetworkX</i> library with reviews as vertices and ranked scores as edges. The [pagerank](https://networkx.org/documentation/stable/reference/algorithms/generated/networkx.algorithms.link_analysis.pagerank_alg.pagerank.html) function, which utilises an algorithm similar to how search engines rank webpages, stack ranks all review nodes against each other. Scores are returned with the proper review node attached, and highest ranking reviews(s) could be considered as the review(s) that best represent(s) the general consensus.

### Part III: Summarisation

#### Processing: Initial

In [23]:
def reviewsFilterByLength(initial_positive_reviews):
    for i in range(len(nms_initial['Reviews'].loc[(nms_initial['Recommended?'] == True)].values)):
        initial_positive_reviews.append(nms_initial['Reviews'].loc[(nms_initial['Recommended?'] == True)].values[i])

    return initial_positive_reviews

def reviewsNLPByBlock(initial_positive_nlp_aggregate, initial_positive_reviews):
    initial_positive_nlp = [initial_positive_reviews[(i*100):(i+1)*100] for i in range(math.ceil(len(initial_positive_reviews)/100))]
    
    for i in range(len(initial_positive_nlp)):
        try:
            initial_positive_nlp_aggregate += generate_summary(initial_positive_nlp[i])
        except:
            continue
    
    return initial_positive_nlp_aggregate

def displayInitialPositiveNLP(initial_positive_nlp_aggregate):
    initial_positive_nlp_aggregate_final = generate_summary(initial_positive_nlp_aggregate)

    for i in range(len(initial_positive_nlp_aggregate_final)):
        print(f'{initial_positive_nlp_aggregate_final[i]}')
        print('---')

def main():
    initial_positive_reviews = []
    initial_positive_nlp_aggregate = []
    filtered_review_list = reviewsFilterByLength(initial_positive_reviews)
    filtered_review_list_nlp = reviewsNLPByBlock(initial_positive_nlp_aggregate, initial_positive_reviews)
    displayInitialPositiveNLP(initial_positive_nlp_aggregate)

if __name__ == "__main__":
    main()

I can't recall a game in recent memory that has had such a large divisive player base. But for me, looking for a game where I can just explore, this is exactly what I wanted. I'm having a blast so far. I do hope they continue to update the game because it would certainly benefit from it. And even though the game runs fine on my computer it still needs optimization and less awkward pop-in textures.
---
One of the best games I have played in a long time. It just came out a day ago at the time of this review and look at my hours. Not sure what other people are whining about, but if you're looking for an exploration game it doesn't get much better than this. Love the randomly generated creatures, sounds, fungi, plants, and planets.
---
It's a simple and fun randomly generated world that is the size of an entire galaxy, I could not ask for more from Hello games. Some people seem to have problems running the game but I have not had any crashes or black screens and I don't have a supercompute

In [24]:
def reviewsFilterByLength(initial_negative_reviews):
    for i in range(len(nms_initial['Reviews'].loc[(nms_initial['Recommended?'] == False)].values)):
        initial_negative_reviews.append(nms_initial['Reviews'].loc[(nms_initial['Recommended?'] == False)].values[i])

    return initial_negative_reviews

def reviewsNLPByBlock(initial_negative_nlp_aggregate, initial_negative_reviews):
    initial_negative_nlp = [initial_negative_reviews[(i*100):(i+1)*100] for i in range(math.ceil(len(initial_negative_reviews)/100))]

    for i in range(len(initial_negative_nlp)):
        try:
            initial_negative_nlp_aggregate += generate_summary(initial_negative_nlp[i])
        except:
            continue
    
    return initial_negative_nlp_aggregate

def displayInitialNegativeNLP(initial_negative_nlp_aggregate):
    initial_negative_nlp_aggregate_final = generate_summary(initial_negative_nlp_aggregate)

    for i in range(len(initial_negative_nlp_aggregate_final)):
        print(f'{initial_negative_nlp_aggregate_final[i]}')
        print('---')

def main():
    initial_negative_reviews = []
    initial_negative_nlp_aggregate = []
    filtered_review_list = reviewsFilterByLength(initial_negative_reviews)
    filtered_review_list_nlp = reviewsNLPByBlock(initial_negative_nlp_aggregate, initial_negative_reviews)
    displayInitialNegativeNLP(initial_negative_nlp_aggregate)

if __name__ == "__main__":
    main()

I bought this game for quite a hefty price being excited for a space survival/ exploration game. I started it up and was excited for all the adventures I will go on. After a while of getting used to the game I learned new recipes and new alien species as I whip around the universe. That's is all I have done for the 40 hours of this game. Lack of content and is boring, not worth the load of money I spent on this game.
---
Was highly looking forward to this, but after playing you soon realise it has very little to offer. The gameplay gets dull very quickly... once youve been to a few planets.. youve pretty much seen and done it all.

This would be perfect as a free download or a bonus game you get free when you buy another game. If theres one tip i can give to people wanting to buy and try this game... please dont, dont end up like me and the others who lost our hard earned money on this piece of trash.


---
The scale of the game is massive and impressive. Other than that huge let down.

#### Processing: Foundation

In [25]:
def reviewsFilterByLength(foundation_positive_reviews):
    for i in range(len(nms_foundation['Reviews'].loc[(nms_foundation['Recommended?'] == True)].values)):
        foundation_positive_reviews.append(nms_foundation['Reviews'].loc[(nms_foundation['Recommended?'] == True)].values[i])

    return foundation_positive_reviews

def reviewsNLPByBlock(foundation_positive_nlp_aggregate, foundation_positive_reviews):
    foundation_positive_nlp = [foundation_positive_reviews[(i*100):(i+1)*100] for i in range(math.ceil(len(foundation_positive_reviews)/100))]

    for i in range(len(foundation_positive_nlp)):
        try:
            foundation_positive_nlp_aggregate += generate_summary(foundation_positive_nlp[i])
        except:
            continue
    
    return foundation_positive_nlp_aggregate

def displayFoundationPositiveNLP(foundation_positive_nlp_aggregate):
    foundation_positive_nlp_aggregate_final = generate_summary(foundation_positive_nlp_aggregate)

    for i in range(len(foundation_positive_nlp_aggregate_final)):
        print(f'{foundation_positive_nlp_aggregate_final[i]}')
        print('---')

def main():
    foundation_positive_reviews = []
    foundation_positive_nlp_aggregate = []
    filtered_review_list = reviewsFilterByLength(foundation_positive_reviews)
    filtered_review_list_nlp = reviewsNLPByBlock(foundation_positive_nlp_aggregate, foundation_positive_reviews)
    displayFoundationPositiveNLP(foundation_positive_nlp_aggregate)

if __name__ == "__main__":
    main()

I was excited for this game when I first heard about it and saw the trailers. However, the game itself dissapointed me: the infinite universe is cool yet gets repetitive; the graphics are breathtaking yet poorly optimised and ultimately there isn't much to do. I want to like this game and I kinda do, but it is definitely not worth full price.

This game is ok but I would only buy at a discount.

5/10
---
Well, Initially, I rated this game a thumbs up, I liked it for what it was...but after months of complete silence I got to thinking the game had been abandoned and changed my review to negative.  This new Foundation update gives me a lot of faith.  Some really cool new featurs and fixes to the game and a good reason to be optimisitic, plus the game runs very smooth on my hardware now.
---
I'm not in general a huge fan of early acess, but games like "the long dark" show an amazing example of how great early acess can be for players and developers. Im sure there would still be complainer

In [26]:
def reviewsFilterByLength(foundation_negative_reviews):
    for i in range(len(nms_foundation['Reviews'].loc[(nms_foundation['Recommended?'] == False)].values)):
        foundation_negative_reviews.append(nms_foundation['Reviews'].loc[(nms_foundation['Recommended?'] == False)].values[i])

    return foundation_negative_reviews

def reviewsNLPByBlock(foundation_negative_nlp_aggregate, foundation_negative_reviews):
    foundation_negative_nlp = [foundation_negative_reviews[(i*100):(i+1)*100] for i in range(math.ceil(len(foundation_negative_reviews)/100))]

    for i in range(len(foundation_negative_nlp)):
        try:
            foundation_negative_nlp_aggregate += generate_summary(foundation_negative_nlp[i])
        except:
            continue
    
    return foundation_negative_nlp_aggregate

def displayFoundationNegativeNLP(foundation_negative_nlp_aggregate):
    foundation_negative_nlp_aggregate_final = generate_summary(foundation_negative_nlp_aggregate)

    for i in range(len(foundation_negative_nlp_aggregate_final)):
        print(f'{foundation_negative_nlp_aggregate_final[i]}')
        print('---')

def main():
    foundation_negative_reviews = []
    foundation_negative_nlp_aggregate = []
    filtered_review_list = reviewsFilterByLength(foundation_negative_reviews)
    filtered_review_list_nlp = reviewsNLPByBlock(foundation_negative_nlp_aggregate, foundation_negative_reviews)
    displayFoundationNegativeNLP(foundation_negative_nlp_aggregate)

if __name__ == "__main__":
    main()

So i made a review when the game came out where i recommended it. Now i defently dont recommend it first of all this game is dead nobody plays it and then all the glitches and the slow gameplay i got all my aroudn 20 hours data just deleted , Baam just like that. When it comes to gameplay you never see other creatures and the graphics is not super good and i,ve heard the ending is pretty boring. i would not recommend it.
---
Nominated for "Biggest Letdown" award.

The game is fun at first, but gets repetitive. It also doesn't deliver on everything that was promised. That being said, I think I got my money's worth, and if you buy this game on sale you can at least get close to what you paid for.

Unless you care more about how pretty some of the worlds can be than the actual gameplay itself, I wouldn't recommend it at full price.
---
No man's sky? more like Big fat lie... not a game worth getting when there are other sandbox games that pull off the exploration vib so much better, Even S

#### Processing: Pathfinder

In [27]:
def reviewsFilterByLength(pathfinder_positive_reviews):
    for i in range(len(nms_pathfinder['Reviews'].loc[(nms_pathfinder['Recommended?'] == True)].values)):
        pathfinder_positive_reviews.append(nms_pathfinder['Reviews'].loc[(nms_pathfinder['Recommended?'] == True)].values[i])

    return pathfinder_positive_reviews

def reviewsNLPByBlock(pathfinder_positive_nlp_aggregate, pathfinder_positive_reviews):
    pathfinder_positive_nlp = [pathfinder_positive_reviews[(i*100):(i+1)*100] for i in range(math.ceil(len(pathfinder_positive_reviews)/100))]

    for i in range(len(pathfinder_positive_nlp)):
        try:
            pathfinder_positive_nlp_aggregate += generate_summary(pathfinder_positive_nlp[i])
        except:
            continue
    
    return pathfinder_positive_nlp_aggregate

def displayPathfinderPositiveNLP(pathfinder_positive_nlp_aggregate):
    pathfinder_positive_nlp_aggregate_final = generate_summary(pathfinder_positive_nlp_aggregate)

    for i in range(len(pathfinder_positive_nlp_aggregate_final)):
        print(f'{pathfinder_positive_nlp_aggregate_final[i]}')
        print('---')

def main():
    pathfinder_positive_reviews = []
    pathfinder_positive_nlp_aggregate = []
    filtered_review_list = reviewsFilterByLength(pathfinder_positive_reviews)
    filtered_review_list_nlp = reviewsNLPByBlock(pathfinder_positive_nlp_aggregate, pathfinder_positive_reviews)
    displayPathfinderPositiveNLP(pathfinder_positive_nlp_aggregate)

if __name__ == "__main__":
    main()

64 hours into this game and I love it. After reading the reviews from release until now opinions seemed to change. The game looks to be more like the promised article. (With more to come). I picked it up in the sale and it's been one of the best games I have played in a while. The pick up and play style is perfect for me as I have to juggle work and family. 
So much to do and strangly relaxing. Can't recommend this enough.
---
Bought this game back when it came out. It was initally a disappointment, but the updates rolling out are more than just patches. They are filling out their game and doing what they can to ammend the complaints of the fans, and the game is very enjoyable at this point. It deserves a shot, especially due to the fact that they are continuing to release large amounts of content that seem to have only made the game better and better thus far. Multiplayer is all I ask for at this point!
---
Since the updates and added content I've really enjoyed this game I'm hoping w

In [28]:
def reviewsFilterByLength(pathfinder_negative_reviews):
    for i in range(len(nms_pathfinder['Reviews'].loc[(nms_pathfinder['Recommended?'] == False)].values)):
        pathfinder_negative_reviews.append(nms_pathfinder['Reviews'].loc[(nms_pathfinder['Recommended?'] == False)].values[i])

    return pathfinder_negative_reviews

def reviewsNLPByBlock(pathfinder_negative_nlp_aggregate, pathfinder_negative_reviews):
    pathfinder_negative_nlp = [pathfinder_negative_reviews[(i*100):(i+1)*100] for i in range(math.ceil(len(pathfinder_negative_reviews)/100))]

    for i in range(len(pathfinder_negative_nlp)):
        try:
            pathfinder_negative_nlp_aggregate += generate_summary(pathfinder_negative_nlp[i])
        except:
            continue
    
    return pathfinder_negative_nlp_aggregate

def displayPathfinderNegativeNLP(pathfinder_negative_nlp_aggregate):
    pathfinder_negative_nlp_aggregate_final = generate_summary(pathfinder_negative_nlp_aggregate)

    for i in range(len(pathfinder_negative_nlp_aggregate_final)):
        print(f'{pathfinder_negative_nlp_aggregate_final[i]}')
        print('---')

def main():
    pathfinder_negative_reviews = []
    pathfinder_negative_nlp_aggregate = []
    filtered_review_list = reviewsFilterByLength(pathfinder_negative_reviews)
    filtered_review_list_nlp = reviewsNLPByBlock(pathfinder_negative_nlp_aggregate, pathfinder_negative_reviews)
    displayPathfinderNegativeNLP(pathfinder_negative_nlp_aggregate)

if __name__ == "__main__":
    main()

This game was so over hyped, followed it from the start and played it for two weeks, wanted to quit after 20 minutes but after spending 60 dollars on it I decided to hold out and give it time hoping it would get better. It didn't, this entire game is all one giant fetch quest. This should be the creators disgrace.
---
once you have traveled from one planet to the next for the first time, BAM thats it, thats the best part of this game. it gets utterly less impressive from there on out. Bought this game early and have been nothing but dissapointed with overall playability of the game. IF you took this graphic engine, and combined it with the gameplay of STARBOUND we would be on to something magical. If you want to just get stoned and look at pretty shit this game is for you
---
Terrible game, completely lied about almost everything to do with this game. I can't believe I paid full price for this game even after playing it ahead of time. Maybe I just thought they would do right by their c

#### Processing: Atlas Rises

In [29]:
def reviewsFilterByLength(ar_positive_reviews):
    for i in range(len(nms_atlas_rises['Reviews'].loc[(nms_atlas_rises['Recommended?'] == True)].values)):
        ar_positive_reviews.append(nms_atlas_rises['Reviews'].loc[(nms_atlas_rises['Recommended?'] == True)].values[i])

    return ar_positive_reviews

def reviewsNLPByBlock(ar_positive_nlp_aggregate, ar_positive_reviews):
    ar_positive_nlp = [ar_positive_reviews[(i*100):(i+1)*100] for i in range(math.ceil(len(ar_positive_reviews)/100))]

    for i in range(len(ar_positive_nlp)):
        try:
            ar_positive_nlp_aggregate += generate_summary(ar_positive_nlp[i])
        except:
            continue
    
    return ar_positive_nlp_aggregate

def displayARPositiveNLP(ar_positive_nlp_aggregate):
    ar_positive_nlp_aggregate_final = generate_summary(ar_positive_nlp_aggregate)

    for i in range(len(ar_positive_nlp_aggregate_final)):
        print(f'{ar_positive_nlp_aggregate_final[i]}')
        print('---')

def main():
    ar_positive_reviews = []
    ar_positive_nlp_aggregate = []
    filtered_review_list = reviewsFilterByLength(ar_positive_reviews)
    filtered_review_list_nlp = reviewsNLPByBlock(ar_positive_nlp_aggregate, ar_positive_reviews)
    displayARPositiveNLP(ar_positive_nlp_aggregate)

if __name__ == "__main__":
    main()

Only my second review of a game in about 5 years and NMS compelled me to do it. I bought the game the day it released and although I was disappointed, I pushed through the fact that features were missing and over the course of about a month I had only played enough to sate my curiosity.

Fast forward a year, and I can't put it down. After the newest patch it evolved into one of the best games I have played and I am now following its development on a regular basis. The future looks bright
---
So I bought this game on launch (for PS4), and like many I was disappointed. After that first week of release, I never touched it again. That was until update 1.3. While Hello Games still have some work to do, it is very exciting to see this kind of progress being made. 

I would highly recommend it if anyone has been on the fence. It is certainly worth the sale price right now.
---
Pre-ordered it day before launch and still don't regret it. Guess it's because I don't cry like a child when somethin

In [30]:
def reviewsFilterByLength(ar_negative_reviews):
    for i in range(len(nms_abyss['Reviews'].loc[(nms_abyss['Recommended?'] == False)].values)):
        ar_negative_reviews.append(nms_abyss['Reviews'].loc[(nms_abyss['Recommended?'] == False)].values[i])

    return ar_negative_reviews

def reviewsNLPByBlock(ar_negative_nlp_aggregate, ar_negative_reviews):
    ar_negative_nlp = [ar_negative_reviews[(i*100):(i+1)*100] for i in range(math.ceil(len(ar_negative_reviews)/100))]

    for i in range(len(ar_negative_nlp)):
        try:
            ar_negative_nlp_aggregate += generate_summary(ar_negative_nlp[i])
        except:
            continue
    
    return ar_negative_nlp_aggregate

def displayARNegativeNLP(ar_negative_nlp_aggregate):
    ar_negative_nlp_aggregate_final = generate_summary(ar_negative_nlp_aggregate)

    for i in range(len(ar_negative_nlp_aggregate_final)):
        print(f'{ar_negative_nlp_aggregate_final[i]}')
        print('---')

def main():
    ar_negative_reviews = []
    ar_negative_nlp_aggregate = []
    filtered_review_list = reviewsFilterByLength(ar_negative_reviews)
    filtered_review_list_nlp = reviewsNLPByBlock(ar_negative_nlp_aggregate, ar_negative_reviews)
    displayARNegativeNLP(ar_negative_nlp_aggregate)

if __name__ == "__main__":
    main()

I wanted to like this game, but found teh User Interface less than intuitive with the quest instruction on how to do ceratin things vague or incomplete (like "press x to access galaxy map" but no explaination of how to navigate to the galaxy map after you press x). It seemed like every new activity took several seconds to a few minutes trying to figure out how to do what the game was telling me to do, praying I figured out how to do it before I died.
---
Figured I would give this game another try after the big update, but frankly, it's still boring as hell. I wish it turned out as exciting as the reveal at E3 back in the day.
---
I've been on Steam for 14 years and own 225 games. This is the first Steam review I have ever written, and also the first time I have requested a refund for a game. I can't think of a better way to turn new players away from your game than a "tutorial" that places you on a planet with aggressively lethal hazards (radiation, toxicity, temperature)  and then giv

#### Processing: NEXT

In [31]:
def reviewsFilterByLength(next_positive_reviews):
    for i in range(len(nms_next['Reviews'].loc[(nms_next['Recommended?'] == True)].values)):
        next_positive_reviews.append(nms_next['Reviews'].loc[(nms_next['Recommended?'] == True)].values[i])

    return next_positive_reviews

def reviewsNLPByBlock(next_positive_nlp_aggregate, next_positive_reviews):
    next_positive_nlp = [next_positive_reviews[(i*100):(i+1)*100] for i in range(math.ceil(len(next_positive_reviews)/100))]

    for i in range(len(next_positive_nlp)):
        try:
            next_positive_nlp_aggregate += generate_summary(next_positive_nlp[i])
        except:
            continue
    
    return next_positive_nlp_aggregate

def displayNEXTPositiveNLP(next_positive_nlp_aggregate):
    next_positive_nlp_aggregate_final = generate_summary(next_positive_nlp_aggregate)

    for i in range(len(next_positive_nlp_aggregate_final)):
        print(f'{next_positive_nlp_aggregate_final[i]}')
        print('---')

def main():
    next_positive_reviews = []
    next_positive_nlp_aggregate = []
    filtered_review_list = reviewsFilterByLength(next_positive_reviews)
    filtered_review_list_nlp = reviewsNLPByBlock(next_positive_nlp_aggregate, next_positive_reviews)
    displayNEXTPositiveNLP(next_positive_nlp_aggregate)

if __name__ == "__main__":
    main()

EDIT: 

Fair play to the Sean and the rest of the guys at Hello Games. After releasing a half baked game they've continued to work on updating it to realise their vision. 

After leaving a scathing review at launch I'm editing this to a Yes I do recommend. Can't say I would buy it at full price, but it's a great experience. This is still pending the NEXT update, I will update this shortly.
---
A stain on the reputation of space exploration games.

EDIT: Wow what a turn around. The devs have finally delivered just about everything they'd promised on the first release. Just consider the last two years as an early access and give this game a shot now, it's a blast now. Definitely pick it up on sale though as I'm still not sure this game is $60
---
I reccomend this game, it has fun gameplay, I don't know what the technical issues may be since I have never ran into them. The game has a lot of replayability for the person who likes to explore. The developers have fufilled their original laun

In [32]:
def reviewsFilterByLength(next_negative_reviews):
    for i in range(len(nms_next['Reviews'].loc[(nms_next['Recommended?'] == False)].values)):
        next_negative_reviews.append(nms_next['Reviews'].loc[(nms_next['Recommended?'] == False)].values[i])

    return next_negative_reviews

def reviewsNLPByBlock(next_negative_nlp_aggregate, next_negative_reviews):
    next_negative_nlp = [next_negative_reviews[(i*100):(i+1)*100] for i in range(math.ceil(len(next_negative_reviews)/100))]

    for i in range(len(next_negative_nlp)):
        try:
            next_negative_nlp_aggregate += generate_summary(next_negative_nlp[i])
        except:
            continue
    
    return next_negative_nlp_aggregate

def displayNEXTNegativeNLP(next_negative_nlp_aggregate):
    next_negative_nlp_aggregate_final = generate_summary(next_negative_nlp_aggregate)

    for i in range(len(next_negative_nlp_aggregate_final)):
        print(f'{next_negative_nlp_aggregate_final[i]}')
        print('---')

def main():
    next_negative_reviews = []
    next_negative_nlp_aggregate = []
    filtered_review_list = reviewsFilterByLength(next_negative_reviews)
    filtered_review_list_nlp = reviewsNLPByBlock(next_negative_nlp_aggregate, next_negative_reviews)
    displayNEXTNegativeNLP(next_negative_nlp_aggregate)

if __name__ == "__main__":
    main()

I wouldn't recommend (but ymmv).
In the end you're waiting a lot in this game. Either in a ditch due to some firestorm on an ice planet, or searching for aliens, or pulling 1k raw material through a refiner to get 500 something else. 
This is the first game that made me quit early on due to the uselessness of it all (compared to many hunderds of hours runescape and the diablos).
But if you like scenery and don't mind waiting, it may appeal to you.
---
This game did not work well with my computer which has run every other game I've played so far just fine. Super laggy and slow, then just cut out completely. Took forever to load. I also found the game to be immediately confusing and was dying very fast just on normal mode. Definitely not a game for me, but seemed like a neat concept from reading about it. The game was a gift, but immediately letting them request a refund so it doesn't go to waste. Thank you for the kind thought, friend!
---
After coming back to this game after it launche

#### Processing: Abyss

In [33]:
def reviewsFilterByLength(abyss_positive_reviews):
    for i in range(len(nms_abyss['Reviews'].loc[(nms_abyss['Recommended?'] == True)].values)):
        abyss_positive_reviews.append(nms_abyss['Reviews'].loc[(nms_abyss['Recommended?'] == True)].values[i])

    return abyss_positive_reviews

def reviewsNLPByBlock(abyss_positive_nlp_aggregate, abyss_positive_reviews):
    abyss_positive_nlp = [abyss_positive_reviews[(i*100):(i+1)*100] for i in range(math.ceil(len(abyss_positive_reviews)/100))]
    
    for i in range(len(abyss_positive_nlp)):
        try:
            abyss_positive_nlp_aggregate += generate_summary(abyss_positive_nlp[i])
        except:
            continue
    
    return abyss_positive_nlp_aggregate

def displayAbyssPositiveNLP(abyss_positive_nlp_aggregate):
    abyss_positive_nlp_aggregate_final = generate_summary(abyss_positive_nlp_aggregate)
    
    for i in range(len(abyss_positive_nlp_aggregate_final)):
        print(f'{abyss_positive_nlp_aggregate_final[i]}')
        print('---')

def main():
    abyss_positive_reviews = []
    abyss_positive_nlp_aggregate = []
    filtered_review_list = reviewsFilterByLength(abyss_positive_reviews)
    filtered_review_list_nlp = reviewsNLPByBlock(abyss_positive_nlp_aggregate, abyss_positive_reviews)
    displayAbyssPositiveNLP(abyss_positive_nlp_aggregate)

if __name__ == "__main__":
    main()

Credit where credit is due, they lived up to their promises albiet 2 years later, a game that was made for a niche market that got marketed for everyone finally came good and is deservedly living up to the hype. Well done Hello Games, thank you so much for not abandoning the game and continuing on with the work I'm sure you love doing.
---
I was only half interested at first in this game, but the Atlas Rises patch convinced me to buy it, and i haven't regretted the decision. The game is really good, if you like a slow exploration game with a light story about existential questions. The stream of patches since NEXT shows a bright future for No Man's Sky. I would consider even buying an expansion now.
---
As we all know it did not exactly manage to get on level with all the hype before its release, it was lacking a lot of things. I still bought the game and I do not regret that decision. I can play for hours without getting bored, seeing new things all the time. Of course it gets dull se

In [34]:
def reviewsFilterByLength(abyss_negative_reviews):
    for i in range(len(nms_abyss['Reviews'].loc[(nms_abyss['Recommended?'] == False)].values)):
        abyss_negative_reviews.append(nms_abyss['Reviews'].loc[(nms_abyss['Recommended?'] == False)].values[i])
    
    return abyss_negative_reviews

def reviewsNLPByBlock(abyss_negative_nlp_aggregate, abyss_negative_reviews):
    abyss_negative_nlp = [abyss_negative_reviews[(i*100):(i+1)*100] for i in range(math.ceil(len(abyss_negative_reviews)/100))]

    for i in range(len(abyss_negative_nlp)):
        try:
            abyss_negative_nlp_aggregate += generate_summary(abyss_negative_nlp[i])
        except:
            continue
        
        return abyss_negative_nlp_aggregate

def displayAbyssNegativeNLP(abyss_negative_nlp_aggregate):
    abyss_negative_nlp_aggregate_final = generate_summary(abyss_negative_nlp_aggregate)

    for i in range(len(abyss_negative_nlp_aggregate_final)):
        print(f'{abyss_negative_nlp_aggregate_final[i]}')
        print('---')

def main():
    abyss_negative_reviews = []
    abyss_negative_nlp_aggregate = []
    filtered_review_list = reviewsFilterByLength(abyss_negative_reviews)
    filtered_review_list_nlp = reviewsNLPByBlock(abyss_negative_nlp_aggregate, abyss_negative_reviews)
    displayAbyssNegativeNLP(abyss_negative_nlp_aggregate)

if __name__ == "__main__":
    main()

I wanted to like this game, but found teh User Interface less than intuitive with the quest instruction on how to do ceratin things vague or incomplete (like "press x to access galaxy map" but no explaination of how to navigate to the galaxy map after you press x). It seemed like every new activity took several seconds to a few minutes trying to figure out how to do what the game was telling me to do, praying I figured out how to do it before I died.
---
Figured I would give this game another try after the big update, but frankly, it's still boring as hell. I wish it turned out as exciting as the reveal at E3 back in the day.
---
I've been on Steam for 14 years and own 225 games. This is the first Steam review I have ever written, and also the first time I have requested a refund for a game. I can't think of a better way to turn new players away from your game than a "tutorial" that places you on a planet with aggressively lethal hazards (radiation, toxicity, temperature)  and then giv

#### Processing: Visions

In [35]:
def reviewsFilterByLength(visions_positive_reviews):
    for i in range(len(nms_visions['Reviews'].loc[(nms_visions['Recommended?'] == True)].values)):
        visions_positive_reviews.append(nms_visions['Reviews'].loc[(nms_visions['Recommended?'] == True)].values[i])

    return visions_positive_reviews

def reviewsNLPByBlock(visions_positive_nlp_aggregate, visions_positive_reviews):
    visions_positive_nlp = [visions_positive_reviews[(i*100):(i+1)*100] for i in range(math.ceil(len(visions_positive_reviews)/50))]

    for i in range(len(visions_positive_nlp)):
        try:
            visions_positive_nlp_aggregate += generate_summary(visions_positive_nlp[i])
        except:
            continue
    
    return visions_positive_nlp_aggregate

def displayVisionsPositiveNLP(visions_positive_nlp_aggregate):
    visions_positive_nlp_aggregate_final = generate_summary(visions_positive_nlp_aggregate)

    for i in range(len(visions_positive_nlp_aggregate_final)):
        print(f'{visions_positive_nlp_aggregate_final[i]}')
        print('---')

def main():
    visions_positive_reviews = []
    visions_positive_nlp_aggregate = []
    filtered_review_list = reviewsFilterByLength(visions_positive_reviews)
    filtered_review_list_nlp = reviewsNLPByBlock(visions_positive_nlp_aggregate, visions_positive_reviews)
    displayVisionsPositiveNLP(visions_positive_nlp_aggregate)

if __name__ == "__main__":
    main()

I have followed this game from the beginning, and I appreciate the work that the devs have put in after such a lackluster launch. It is now more fleshed out; It is what was promised in the beginning and so much more. It still may not be for everyone, but it is a very nice game for anyone who likes to explore the universe.
---
I bought this game on release day and was entertained but I did share the same sentiment of disappointment as many others. I came back to it a week ago and with all the new patches and updates it has a lot of new life. Even after such a discouraging launch, the devs at Hello Games still worked on making things better and I sincerely respect that. I'm looking forward to this summer's update! This game is literally open-universe and I highly recommend getting a copy
---
#Rewriting this review almost three years later. Played 20 hours on release, haven't touched it until two days ago and I've been playing it heaps.

The content and sense of progression has been so we

In [36]:
def reviewsFilterByLength(visions_negative_reviews):
    for i in range(len(nms_visions['Reviews'].loc[(nms_visions['Recommended?'] == False)].values)):
        visions_negative_reviews.append(nms_visions['Reviews'].loc[(nms_visions['Recommended?'] == False)].values[i])
    
    return visions_negative_reviews

def reviewsNLPByBlock(visions_negative_nlp_aggregate, visions_negative_reviews):
    visions_negative_nlp = [visions_negative_reviews[(i*100):(i+1)*100] for i in range(math.ceil(len(visions_negative_reviews)/100))]

    for i in range(len(visions_negative_nlp)):
        try:
            visions_negative_nlp_aggregate += generate_summary(visions_negative_nlp[i])
        except:
            continue
        
        return visions_negative_nlp_aggregate

def displayVisionsNegativeNLP(visions_negative_nlp_aggregate):
    visions_negative_nlp_aggregate_final = generate_summary(visions_negative_nlp_aggregate)

    for i in range(len(visions_negative_nlp_aggregate_final)):
        print(f'{visions_negative_nlp_aggregate_final[i]}')
        print('---')

def main():
    visions_negative_reviews = []
    visions_negative_nlp_aggregate = []
    filtered_review_list = reviewsFilterByLength(visions_negative_reviews)
    filtered_review_list_nlp = reviewsNLPByBlock(visions_negative_nlp_aggregate, visions_negative_reviews)
    displayVisionsNegativeNLP(visions_negative_nlp_aggregate)

if __name__ == "__main__":
    main()

After about a week i had seen all this game has to offer at this point in time. I continued playing though as i had some goals i set for myself. And now i am at the center of the galaxy and quite frankly i am disappointed. Procederually generated sameness. Same creatures, planets, system layouts. Over and over. Even building my base bored me. This game appeals to a certain type of gamer. I thought i was one of them, turns out i am not.
---
It's a good engine poorly executed. It should be called No Purpose Sky. At first it felt like a poor mans version of Subnutica however after a bunch of grinding it just seemed like it stole the concepts but didn't deliver the game playing experience in regards to a story line driven plot that is immersive and compelling.  If you love space shooter that's only reason is grinding then this game is for you! Otherwise just avoid this steaming pile of.... 

just don't... you'll understand if you do.
---
Only tried to play this game because i seen multipla

#### Processing: Beyond

In [37]:
def reviewsFilterByLength(beyond_positive_reviews):
    for i in range(len(nms_beyond['Reviews'].loc[(nms_beyond['Recommended?'] == True)].values)):
        beyond_positive_reviews.append(nms_beyond['Reviews'].loc[(nms_beyond['Recommended?'] == True)].values[i])

    return beyond_positive_reviews

def reviewsNLPByBlock(beyond_positive_nlp_aggregate, beyond_positive_reviews):
    beyond_positive_nlp = [beyond_positive_reviews[(i*100):(i+1)*100] for i in range(math.ceil(len(beyond_positive_reviews)/100))]

    for i in range(len(beyond_positive_nlp)):
        try:
            beyond_positive_nlp_aggregate += generate_summary(beyond_positive_nlp[i])
        except:
            continue
    
    return beyond_positive_nlp_aggregate

def displayBeyondPositiveNLP(beyond_positive_nlp_aggregate):
    beyond_positive_nlp_aggregate_final = generate_summary(beyond_positive_nlp_aggregate)

    for i in range(len(beyond_positive_nlp_aggregate_final)):
        print(f'{beyond_positive_nlp_aggregate_final[i]}')
        print('---')

def main():
    beyond_positive_reviews = []
    beyond_positive_nlp_aggregate = []
    filtered_review_list = reviewsFilterByLength(beyond_positive_reviews)
    filtered_review_list_nlp = reviewsNLPByBlock(beyond_positive_nlp_aggregate, beyond_positive_reviews)
    displayBeyondPositiveNLP(beyond_positive_nlp_aggregate)

if __name__ == "__main__":
    main()

I was never on the hype train for this game so im not really dissapointed. And im actually very proud of hello games, they faced alot of hate and there still bringing out free updates. That is some respectable commitment. From what ive seen the game is great and very unique. Just dont play it with intel. Intel drivers do not a good graphics make
---
I love this game and it's worth the money but whoever managed the deployment of the last major upgrade should be ashamed. I was unable to play the game for about 2 weeks because I cannot remap my keyboard configuration with the new version. It's like they want to force everyone onto a game controller and console and well, no thank you. I'm a PC Gamer for life.
---
I figure it is about time that I review this game now that I am close to almost 200 hours. When this game came out, it was disappointing, and I did stop playing it for awhile.  But the recent updates and improvements have made this game miles and miles better. If you've heard bad 

In [38]:
def reviewsFilterByLength(beyond_negative_reviews):
    for i in range(len(nms_beyond['Reviews'].loc[(nms_beyond['Recommended?'] == False)].values)):
        beyond_negative_reviews.append(nms_beyond['Reviews'].loc[(nms_beyond['Recommended?'] == False)].values[i])
    
    return beyond_negative_reviews

def reviewsNLPByBlock(beyond_negative_nlp_aggregate, beyond_negative_reviews):
    beyond_negative_nlp = [beyond_negative_reviews[(i*100):(i+1)*100] for i in range(math.ceil(len(beyond_negative_reviews)/100))]

    for i in range(len(beyond_negative_nlp)):
        try:
            beyond_negative_nlp_aggregate += generate_summary(beyond_negative_nlp[i])
        except:
            continue
        
        return beyond_negative_nlp_aggregate

def displayBeyondNegativeNLP(beyond_negative_nlp_aggregate):
    beyond_negative_nlp_aggregate_final = generate_summary(beyond_negative_nlp_aggregate)

    for i in range(len(beyond_negative_nlp_aggregate_final)):
        print(f'{beyond_negative_nlp_aggregate_final[i]}')
        print('---')

def main():
    beyond_negative_reviews = []
    beyond_negative_nlp_aggregate = []
    filtered_review_list = reviewsFilterByLength(beyond_negative_reviews)
    filtered_review_list_nlp = reviewsNLPByBlock(beyond_negative_nlp_aggregate, beyond_negative_reviews)
    displayBeyondNegativeNLP(beyond_negative_nlp_aggregate)

if __name__ == "__main__":
    main()

So, I just crashed into a trading post while trying to take off. It was an unavoidable glitch caused by bad design. After my ship was destroyed, I found that my entire ship's cargo, representing dozens of hours of work, had been erased in the explosion. Basically, I am never wasting my time in this piece of garbage ever again. Three years after launch it's still a barely playable glitchy mess. Uninstalling now, and nothing will ever tempt me back.
---
Not a good VR port, in my limited time with it. Struggled with the controls while crossing pointlessly vast distances in a muddy looking desert to collect arbitrary key items, never managed to get the ship off the ground before I started getting a little sim-sick feeling, which was all the excuse I needed to quit out and refund.
---
I love this game and what it offers, but no matter how much i love it i can't forgive the terrain pop in. This is the worst issue in the game by far and it COMPLETELY ruins the experience of the game. I want t

#### Processing: Synthesis

In [39]:
def reviewsFilterByLength(synthesis_positive_reviews):
    for i in range(len(nms_synthesis['Reviews'].loc[(nms_synthesis['Recommended?'] == True)].values)):
        synthesis_positive_reviews.append(nms_synthesis['Reviews'].loc[(nms_synthesis['Recommended?'] == True)].values[i])

    return synthesis_positive_reviews

def reviewsNLPByBlock(synthesis_positive_nlp_aggregate, synthesis_positive_reviews):
    synthesis_positive_nlp = [synthesis_positive_reviews[(i*100):(i+1)*100] for i in range(math.ceil(len(synthesis_positive_reviews)/100))]

    for i in range(len(synthesis_positive_nlp)):
        try:
            synthesis_positive_nlp_aggregate += generate_summary(synthesis_positive_nlp[i])
        except:
            continue
    
    return synthesis_positive_nlp_aggregate

def displaySynthesisPositiveNLP(synthesis_positive_nlp_aggregate):
    synthesis_positive_nlp_aggregate_final = generate_summary(synthesis_positive_nlp_aggregate)

    for i in range(len(synthesis_positive_nlp_aggregate_final)):
        print(f'{synthesis_positive_nlp_aggregate_final[i]}')
        print('---')

def main():
    synthesis_positive_reviews = []
    synthesis_positive_nlp_aggregate = []
    filtered_review_list = reviewsFilterByLength(synthesis_positive_reviews)
    filtered_review_list_nlp = reviewsNLPByBlock(synthesis_positive_nlp_aggregate, synthesis_positive_reviews)
    displaySynthesisPositiveNLP(synthesis_positive_nlp_aggregate)

if __name__ == "__main__":
    main()

Played from the beginning and holy cow this game has come far! I love the experience. The only thing I'd like to see is more of a story. What I mean by that is like what you have in your classic RPG games such as Final Fantasy and God of War series. The reason is because after you get to a certain point you think "well, now what do I do?" Unfortunately, it doesn't take that relatively long to get there either.
---
I originally played this for 3 days straight when this game first released. At that point the game was fun but got very boring very quickly. In its current state the game is a lot more fun and diverse and there is a lot more to do then when it first came out. I will say that the game can get stale every so often but there are continued updates to the game and it doesn't look like they plan on stopping anytime soon.
---
Boy this game has come a long way. Base building is a lot of fun once you get the hang of it and getting upgraded tech for your ship and base to leverage solar

In [40]:
def reviewsFilterByLength(synthesis_negative_reviews):
    for i in range(len(nms_synthesis['Reviews'].loc[(nms_synthesis['Recommended?'] == False)].values)):
        synthesis_negative_reviews.append(nms_synthesis['Reviews'].loc[(nms_synthesis['Recommended?'] == False)].values[i])
    
    return synthesis_negative_reviews

def reviewsNLPByBlock(synthesis_negative_nlp_aggregate, synthesis_negative_reviews):
    synthesis_negative_nlp = [synthesis_negative_reviews[(i*100):(i+1)*100] for i in range(math.ceil(len(synthesis_negative_reviews)/100))]

    for i in range(len(synthesis_negative_nlp)):
        try:
            synthesis_negative_nlp_aggregate += generate_summary(synthesis_negative_nlp[i])
        except:
            continue
        
    return synthesis_negative_nlp_aggregate

def displaySynthesisNegativeNLP(synthesis_negative_nlp_aggregate):
    synthesis_negative_nlp_aggregate_final = generate_summary(synthesis_negative_nlp_aggregate)

    for i in range(len(synthesis_negative_nlp_aggregate_final)):
        print(f'{synthesis_negative_nlp_aggregate_final[i]}')
        print('---')

def main():
    synthesis_negative_reviews = []
    synthesis_negative_nlp_aggregate = []
    filtered_review_list = reviewsFilterByLength(synthesis_negative_reviews)
    filtered_review_list_nlp = reviewsNLPByBlock(synthesis_negative_nlp_aggregate, synthesis_negative_reviews)
    displaySynthesisNegativeNLP(synthesis_negative_nlp_aggregate)

if __name__ == "__main__":
    main()

I've never played something that felt like it should be so cool, but was so not fun at the same time. Not even including constant game breaking bugs, but the actual game itself just is not fun. Grind materials so you can grind different materials, while we constantly throw un-fun enemies at you. I've repeatedly lost progress due to weird bugs and after nearly two days of trying to get a group past the "tutorial" part, I randomly was hit with another bug that caused me to drop out of the quest.
---
I respect Hello Games for working hard on the disappointment they gave the world to fulfill their promises. Despite this I can not recommend it for its $30-60 price tag. It is still empty and rather soulless. The game play is just grinding and wandering around. Base building is both pointless and useless, so despite it not being that bad you have no end goal besides some points in the story making you use build functions, and a resource sink for all that grinding you do.
---
Pretty sad when y

#### Processing: Living Ship

In [41]:
def reviewsFilterByLength(living_ship_positive_reviews):
    for i in range(len(nms_living_ship['Reviews'].loc[(nms_living_ship['Recommended?'] == True)].values)):
        living_ship_positive_reviews.append(nms_living_ship['Reviews'].loc[(nms_living_ship['Recommended?'] == True)].values[i])

    return living_ship_positive_reviews

def reviewsNLPByBlock(living_ship_positive_nlp_aggregate, living_ship_positive_reviews):
    living_ship_positive_nlp = [living_ship_positive_reviews[(i*100):(i+1)*100] for i in range(math.ceil(len(living_ship_positive_reviews)/100))]

    for i in range(len(living_ship_positive_nlp)):
        try:
            living_ship_positive_nlp_aggregate += generate_summary(living_ship_positive_nlp[i])
        except:
            continue
    
    return living_ship_positive_nlp_aggregate

def displayLivingShipPositiveNLP(living_ship_positive_nlp_aggregate):
    living_ship_positive_nlp_aggregate_final = generate_summary(living_ship_positive_nlp_aggregate)

    for i in range(len(living_ship_positive_nlp_aggregate_final)):
        print(f'{living_ship_positive_nlp_aggregate_final[i]}')
        print('---')

def main():
    living_ship_positive_reviews = []
    living_ship_positive_nlp_aggregate = []
    filtered_review_list = reviewsFilterByLength(living_ship_positive_reviews)
    filtered_review_list_nlp = reviewsNLPByBlock(living_ship_positive_nlp_aggregate, living_ship_positive_reviews)
    displayLivingShipPositiveNLP(living_ship_positive_nlp_aggregate)

if __name__ == "__main__":
    main()

c:\users\ken\appdata\local\programs\python\python38\lib\site-packages\nltk\cluster\util.py:130: RuntimeWarning: invalid value encountered in divide
  return 1 - (numpy.dot(u, v) / (sqrt(numpy.dot(u, u)) * sqrt(numpy.dot(v, v))))


The greatest comeback story ever told. This game has little in the way of story, but a lot in creativity and exploration. There are all sorts of major and minor upgrades to bases, ships, vehicles, weapons, and your suit that take a lot of time and investment. 

Theres so much to do that the lack of a story isnt a big deal if you enjoy simply building, upgrading, collecting and min/maxing. Updates happen every 1-2 months so theres always new stuff to keep you interested.
---
The best way to play this game is not to aim to "beat" the game or "complete" it.  This game gives you an entire universe to explore with lore and cultures that you will enjoy trying to better understand.  The aim of this game is to experience as much as possible of what this virtual universe has to offer.

I've been playing this game since it launched in 2016 and no other game has held my attention for this long.  It's an absolutely fantastic game.
---
After a lot of updates since the initial release this game now 

In [42]:
def reviewsFilterByLength(living_ship_negative_reviews):
    for i in range(len(nms_living_ship['Reviews'].loc[(nms_living_ship['Recommended?'] == False)].values)):
        living_ship_negative_reviews.append(nms_living_ship['Reviews'].loc[(nms_living_ship['Recommended?'] == False)].values[i])
    
    return living_ship_negative_reviews

def reviewsNLPByBlock(living_ship_negative_nlp_aggregate, living_ship_negative_reviews):
    living_ship_negative_nlp = [living_ship_negative_reviews[(i*100):(i+1)*100] for i in range(math.ceil(len(living_ship_negative_reviews)/100))]

    for i in range(len(living_ship_negative_nlp)):
        try:
            living_ship_negative_nlp_aggregate += generate_summary(living_ship_negative_nlp[i])
        except:
            continue
        
    return living_ship_negative_nlp_aggregate

def displayLivingShipNegativeNLP(living_ship_negative_nlp_aggregate):
    living_ship_negative_nlp_aggregate_final = generate_summary(living_ship_negative_nlp_aggregate)

    for i in range(len(living_ship_negative_nlp_aggregate_final)):
        print(f'{living_ship_negative_nlp_aggregate_final[i]}')
        print('---')

def main():
    living_ship_negative_reviews = []
    living_ship_negative_nlp_aggregate = []
    filtered_review_list = reviewsFilterByLength(living_ship_negative_reviews)
    filtered_review_list_nlp = reviewsNLPByBlock(living_ship_negative_nlp_aggregate, living_ship_negative_reviews)
    displayLivingShipNegativeNLP(living_ship_negative_nlp_aggregate)

if __name__ == "__main__":
    main()

This game is pretty, but games like ARK are just as pretty but are more fun. The graphics are about where the fun stops. No Man's Sky is boring when it's not glitching on you. It also doesn't run well in Big Picture Mode with a Steam Controller. Skip this game and spend your money on something else.
---
Fix multiplayer missions. The depot raid glitches out if you don't land at the depot first and instead just destroy it or if your other players have already destroyed it, you cannot complete it. Also the weekly mission where you process the plants. its so fucking annoying. I got 20 hexaberries. but then all of a sudden it wants 20 sweetroot. I finally did that and then when i alt tabbed while in the nutrient processor. boom, it disappeared, along with the roots too  >:  <
---
NOT RECOMMENDED FOR VR ONLY

This is just to express my gripe with the fact that the UI is independent of the headset. If you physically turn 90 degrees -  your health, shield, and mission indicators are now on you

#### Processing: Exo Mech

In [43]:
def reviewsFilterByLength(exo_mech_positive_reviews):
    for i in range(len(nms_exo_mech['Reviews'].loc[(nms_exo_mech['Recommended?'] == True)].values)):
        exo_mech_positive_reviews.append(nms_exo_mech['Reviews'].loc[(nms_exo_mech['Recommended?'] == True)].values[i])

    return exo_mech_positive_reviews

def reviewsNLPByBlock(exo_mech_positive_nlp_aggregate, exo_mech_positive_reviews):
    exo_mech_positive_nlp = [exo_mech_positive_reviews[(i*100):(i+1)*100] for i in range(math.ceil(len(exo_mech_positive_reviews)/100))]

    for i in range(len(exo_mech_positive_nlp)):
        try:
            exo_mech_positive_nlp_aggregate += generate_summary(exo_mech_positive_nlp[i])
        except:
            continue
        
        return exo_mech_positive_nlp_aggregate

def displayExoMechPositiveNLP(exo_mech_positive_nlp_aggregate):
    exo_mech_positive_nlp_aggregate_final = generate_summary(exo_mech_positive_nlp_aggregate)

    for i in range(len(exo_mech_positive_nlp_aggregate_final)):
        print(f'{exo_mech_positive_nlp_aggregate_final[i]}')
        print('---')

def main():
    exo_mech_positive_reviews = []
    exo_mech_positive_nlp_aggregate = []
    filtered_review_list = reviewsFilterByLength(exo_mech_positive_reviews)
    filtered_review_list_nlp = reviewsNLPByBlock(exo_mech_positive_nlp_aggregate, exo_mech_positive_reviews)
    displayExoMechPositiveNLP(exo_mech_positive_nlp_aggregate)

if __name__ == "__main__":
    main()

After all the uproar of the launch of this game I never would have thought I'd buy it. Saw some interesting commentary lately and decided to give it a try. Wow.... the developer has made up for a lot! The amount of new content they have released, for free, is amazing. I'm going to be spending a lot of time on here, well worth the money now.
---
When this game first came out the reviews were terrible. Here i am a few years later finally decided to give it a shot since it was on sale. It's much better now. I still wouldn't say it's perfect, but what game is? It's no longer stitched together with scotch tape and failing to deliver. If you're into the exploring and gathering  and building kinda game, go for it.
---
While the game is quite a grind, it is a fun grind where you get to see the beauty of each and every system, planet, and moon. I really commend Hello Games for fixing the game instead of just moving on to another venture. The game is worth the $60 price tag and it is extremely i

In [44]:
def reviewsFilterByLength(exo_mech_negative_reviews):
    for i in range(len(nms_exo_mech['Reviews'].loc[(nms_exo_mech['Recommended?'] == False)].values)):
        exo_mech_negative_reviews.append(nms_exo_mech['Reviews'].loc[(nms_exo_mech['Recommended?'] == False)].values[i])
    
    return exo_mech_negative_reviews

def reviewsNLPByBlock(exo_mech_negative_nlp_aggregate, exo_mech_negative_reviews):
    exo_mech_negative_nlp = [exo_mech_negative_reviews[(i*100):(i+1)*100] for i in range(math.ceil(len(exo_mech_negative_reviews)/100))]

    for i in range(len(exo_mech_negative_nlp)):
        try:
            exo_mech_negative_nlp_aggregate += generate_summary(exo_mech_negative_nlp[i])
        except:
            continue
        
    return exo_mech_negative_nlp_aggregate

def displayExoMechNegativeNLP(exo_mech_negative_nlp_aggregate):
    exo_mech_negative_nlp_aggregate_final = generate_summary(exo_mech_negative_nlp_aggregate)

    for i in range(len(exo_mech_negative_nlp_aggregate_final)):
        print(f'{exo_mech_negative_nlp_aggregate_final[i]}')
        print('---')

def main():
    exo_mech_negative_reviews = []
    exo_mech_negative_nlp_aggregate = []
    filtered_review_list = reviewsFilterByLength(exo_mech_negative_reviews)
    filtered_review_list_nlp = reviewsNLPByBlock(exo_mech_negative_nlp_aggregate, exo_mech_negative_reviews)
    displayExoMechNegativeNLP(exo_mech_negative_nlp_aggregate)

if __name__ == "__main__":
    main()

Once you get past the hump of how to make money, there's absolutely no point to the game.  No new or cool thing to be found, as its all procedurally created.  All you get by exploring is different generations per planet looking for the combination you want for your base, and that assumes you care that much about aesthetics.
---
Yea its complete trash just like when it first came out(own it on PS4) feels tidius and its a pretty boring grindy game..you can play for 3 hours and feel like you accomplished nothing, DO NOT BUY NO MATTER WHAT THE PRICE. THE inventory system is terrible.
---
Seems like a good game on first impression, but to be honest i wouldn't recommend if playing with friends, the coop experience is buggy with the crucial issue being that you don't share objectives?!?! I don't see how this is coop if your simply playing alongside your friends but not with them you dont really do anything together and cant share a base? Only good for single player and should not be listed as

#### Processing: Crossplay

In [45]:
def reviewsFilterByLength(crossplay_positive_reviews):
    for i in range(len(nms_crossplay['Reviews'].loc[(nms_crossplay['Recommended?'] == True)].values)):
        crossplay_positive_reviews.append(nms_crossplay['Reviews'].loc[(nms_crossplay['Recommended?'] == True)].values[i])

    return crossplay_positive_reviews

def reviewsNLPByBlock(crossplay_positive_nlp_aggregate, crossplay_positive_reviews):
    crossplay_positive_nlp = [crossplay_positive_reviews[(i*100):(i+1)*100] for i in range(math.ceil(len(crossplay_positive_reviews)/100))]

    for i in range(len(crossplay_positive_nlp)):
        try:
            crossplay_positive_nlp_aggregate += generate_summary(crossplay_positive_nlp[i])
        except:
            continue
    
    return crossplay_positive_nlp_aggregate

def displayCrossplayPositiveNLP(crossplay_positive_nlp_aggregate):
    crossplay_positive_nlp_aggregate_final = generate_summary(crossplay_positive_nlp_aggregate)

    for i in range(len(crossplay_positive_nlp_aggregate_final)):
        print(f'{crossplay_positive_nlp_aggregate_final[i]}')
        print('---')

def main():
    crossplay_positive_reviews = []
    crossplay_positive_nlp_aggregate = []
    filtered_review_list = reviewsFilterByLength(crossplay_positive_reviews)
    filtered_review_list_nlp = reviewsNLPByBlock(crossplay_positive_nlp_aggregate, crossplay_positive_reviews)
    displayCrossplayPositiveNLP(crossplay_positive_nlp_aggregate)

if __name__ == "__main__":
    main()

The game had a very rocky start during release, but I'm happy to say that Hello Games has done an amazing job providing free updates with tons of improvements to the game. Definitely one of my favourite games. The recently added crossplay is brilliant too! I can now play with my friends who own consoles. Great game.
---
I have owned this twice now and the first time I owned it was at release and it was a terrible mess that had me captivated for maybe 10 hours total but.... THIS IS NOT THE SAME GAME ANYMORE! I would recommend anyone get this game especially if your playing in VR.

THANK YOU hello agmes for FINALLY giving us the product we has hoped for, a space sandbox where you can make your mark in the universe and do whatever the hell you please!
---
Greatest comeback in gaming history. No Man's sky started off as a meme, rightfully so, but they turned the ship around and delivered an exciting, enjoyable gameplay experience. There is so much to do in this game, the universe is truly 

In [46]:
def reviewsFilterByLength(crossplay_negative_reviews):
    for i in range(len(nms_crossplay['Reviews'].loc[(nms_crossplay['Recommended?'] == False)].values)):
        crossplay_negative_reviews.append(nms_crossplay['Reviews'].loc[(nms_crossplay['Recommended?'] == False)].values[i])
    
    return crossplay_negative_reviews

def reviewsNLPByBlock(crossplay_negative_nlp_aggregate, crossplay_negative_reviews):
    crossplay_negative_nlp = [crossplay_negative_reviews[(i*100):(i+1)*100] for i in range(math.ceil(len(crossplay_negative_reviews)/100))]

    for i in range(len(crossplay_negative_nlp)):
        try:
            crossplay_negative_nlp_aggregate += generate_summary(crossplay_negative_nlp[i])
        except:
            continue
        
    return crossplay_negative_nlp_aggregate

def displayCrossplayNegativeNLP(crossplay_negative_nlp_aggregate):
    crossplay_negative_nlp_aggregate_final = generate_summary(crossplay_negative_nlp_aggregate)

    for i in range(len(crossplay_negative_nlp_aggregate_final)):
        print(f'{crossplay_negative_nlp_aggregate_final[i]}')
        print('---')

def main():
    crossplay_negative_reviews = []
    crossplay_negative_nlp_aggregate = []
    filtered_review_list = reviewsFilterByLength(crossplay_negative_reviews)
    filtered_review_list_nlp = reviewsNLPByBlock(crossplay_negative_nlp_aggregate, crossplay_negative_reviews)
    displayCrossplayNegativeNLP(crossplay_negative_nlp_aggregate)

if __name__ == "__main__":
    main()

Generally speaking, grinding is fine. It is a legitimate mechanism in a game.
But just to walk around a planet, I am required to mine at least two resources [to receive protection against radiation and oxygen] non-stop ? This is way too much. I saw the catastrophy incoming, and refunded swiftly, before I accumulated 120 minutes of playtime. Also there are too many Asteroids in space. Just as if space would be some kind of building debris trashyard.
---
Game is absolutely not designed for PC. The UI is very strange, requiring you to click and HOLD your mouse button to click menu buttons. Despite this it does NOT work with gamepads or flight sticks. It's a wonderful idea for a game, and it's definitely had some good updates, but its implementation absolutely fails to deliver and isn't a game I can recommend.
---
It's strange for me to type this but I really prefered this game more when it first came out. It was more about lonely exploration then and that's why I purchased it. Now even th

#### Processing: Desolation

In [47]:
def reviewsFilterByLength(desolation_positive_reviews):
    for i in range(len(nms_desolation['Reviews'].loc[(nms_desolation['Recommended?'] == True)].values)):
        desolation_positive_reviews.append(nms_desolation['Reviews'].loc[(nms_desolation['Recommended?'] == True)].values[i])

    return desolation_positive_reviews

def reviewsNLPByBlock(desolation_positive_nlp_aggregate, desolation_positive_reviews):
    desolation_positive_nlp = [desolation_positive_reviews[(i*100):(i+1)*100] for i in range(math.ceil(len(desolation_positive_reviews)/100))]

    for i in range(len(desolation_positive_nlp)):
        try:
            desolation_positive_nlp_aggregate += generate_summary(desolation_positive_nlp[i])
        except:
            continue
    
    return desolation_positive_nlp_aggregate

def displayDesolationPositiveNLP(desolation_positive_nlp_aggregate):
    desolation_positive_nlp_aggregate_final = generate_summary(desolation_positive_nlp_aggregate)

    for i in range(len(desolation_positive_nlp_aggregate_final)):
        print(f'{desolation_positive_nlp_aggregate_final[i]}')
        print('---')

def main():
    desolation_positive_reviews = []
    desolation_positive_nlp_aggregate = []
    filtered_review_list = reviewsFilterByLength(desolation_positive_reviews)
    filtered_review_list_nlp = reviewsNLPByBlock(desolation_positive_nlp_aggregate, desolation_positive_reviews)
    displayDesolationPositiveNLP(desolation_positive_nlp_aggregate)

if __name__ == "__main__":
    main()

This game may still have a ways to go in terms of planet/biome variation, but Hello Games has made an amazing comeback with No Man's Sky when most companies would abandon the project for future endeavors. I play the game in bursts every few months as the new updates roll out, and I find the game to be very enjoyable and relaxing. It also comes with a free VR option that most companies require you to purchase separately.
---
Honestly, for the 4 hours i have played (reviewed at the time) this game is actually fun. its turned from this big let down, to one of the best open space games i have ever played. i can see myself putting constant hours into this game just to see the worlds i can explore, find and make my own bases on it. i really already like this game and it is worth the price it is at the moment. thank you Hello Games for not giving up, and keep doing what you are doing!
---
Despite what was a clear-cut disaster at launch, Hello Games has gone above and beyond and has turned a d

In [48]:
def reviewsFilterByLength(desolation_negative_reviews):
    for i in range(len(nms_desolation['Reviews'].loc[(nms_desolation['Recommended?'] == False)].values)):
        desolation_negative_reviews.append(nms_desolation['Reviews'].loc[(nms_desolation['Recommended?'] == False)].values[i])
    
    return desolation_negative_reviews

def reviewsNLPByBlock(desolation_negative_nlp_aggregate, desolation_negative_reviews):
    desolation_negative_nlp = [desolation_negative_reviews[(i*100):(i+1)*100] for i in range(math.ceil(len(desolation_negative_reviews)/100))]

    for i in range(len(desolation_negative_nlp)):
        try:
            desolation_negative_nlp_aggregate += generate_summary(desolation_negative_nlp[i])
        except:
            continue
        
    return desolation_negative_nlp_aggregate

def displayDesolationNegativeNLP(desolation_negative_nlp_aggregate):
    desolation_negative_nlp_aggregate_final = generate_summary(desolation_negative_nlp_aggregate)

    for i in range(len(desolation_negative_nlp_aggregate_final)):
        print(f'{desolation_negative_nlp_aggregate_final[i]}')
        print('---')

def main():
    desolation_negative_reviews = []
    desolation_negative_nlp_aggregate = []
    filtered_review_list = reviewsFilterByLength(desolation_negative_reviews)
    filtered_review_list_nlp = reviewsNLPByBlock(desolation_negative_nlp_aggregate, desolation_negative_reviews)
    displayDesolationNegativeNLP(desolation_negative_nlp_aggregate)

if __name__ == "__main__":
    main()

Decent game. Going from planet side to space in one motion is great. This isn't a space sim by any means and you will be mining a lot of resources as you build new stuff like a base or equipment that push the story forward. Unfortunately the game is still glitchy, some elements are above ground or sticking out of it while other times the game just crashes outright.
---
It has been a few years since I last played. Some of the changes since then look promising, but I find "Normal" mode extremely difficult and frustrating now. I had to abandon my first 2 replays because, unlike previously, you no longer start next to your ship. Couldn't find it in time and died due to environmental hazards. Managed to progress enough in my 3rd playthrough to find my ship. Until I can fine-tune the twitchy mouse controls, I'll have to limit myself to Creative mode for now.
---
I thought I'd give it a go as I found a cheap key. So far it's ok, no better than that. The world feels empty, devoid of genuine li

#### Processing: Origins

In [49]:
def reviewsFilterByLength(origins_positive_reviews):
    for i in range(len(nms_origins['Reviews'].loc[(nms_origins['Recommended?'] == True)].values)):
        origins_positive_reviews.append(nms_origins['Reviews'].loc[(nms_origins['Recommended?'] == True)].values[i])

    return origins_positive_reviews

def reviewsNLPByBlock(origins_positive_nlp_aggregate, origins_positive_reviews):
    origins_positive_nlp = [origins_positive_reviews[(i*100):(i+1)*100] for i in range(math.ceil(len(origins_positive_reviews)/100))]

    for i in range(len(origins_positive_nlp)):
        try:
            origins_positive_nlp_aggregate += generate_summary(origins_positive_nlp[i])
        except:
            continue
    
    return origins_positive_nlp_aggregate

def displayOriginsPositiveNLP(origins_positive_nlp_aggregate):
    origins_positive_nlp_aggregate_final = generate_summary(origins_positive_nlp_aggregate)

    for i in range(len(origins_positive_nlp_aggregate_final)):
        print(f'{origins_positive_nlp_aggregate_final[i]}')
        print('---')

def main():
    origins_positive_reviews = []
    origins_positive_nlp_aggregate = []
    filtered_review_list = reviewsFilterByLength(origins_positive_reviews)
    filtered_review_list_nlp = reviewsNLPByBlock(origins_positive_nlp_aggregate, origins_positive_reviews)
    displayOriginsPositiveNLP(origins_positive_nlp_aggregate)

if __name__ == "__main__":
    main()

This game is one of the best if not the best crafting/survival game I have ever played. If you like space, exploring, story or just base building this game is for you. Even when I play with friends we can either team up and do whatever or we could do our own thing while still talking and hanging out. The possibilities are endless in this game!
---
No Man's Sky is a really good game. It is so fun flying around in your starship, discovering planets, seeing all the weird and wonderful creatures they have been generated. I skipped this game when it launched because it seemed like a hot mess but I'm so glad that I have recently bought it, 4 years later . Hello Games have done such an amazing job of supporting the game and they should be so proud of the game it is today.
---
I have lots of fun playing No Man's Sky i love the ship mechanics and flying them around, I also like exploring new planets, but there is a bug that randomly crashes the game for me. since i got a better computer after a

In [50]:
def reviewsFilterByLength(origins_negative_reviews):
    for i in range(len(nms_origins['Reviews'].loc[(nms_origins['Recommended?'] == False)].values)):
        origins_negative_reviews.append(nms_origins['Reviews'].loc[(nms_origins['Recommended?'] == False)].values[i])
    
    return origins_negative_reviews

def reviewsNLPByBlock(origins_negative_nlp_aggregate, origins_negative_reviews):
    origins_negative_nlp = [origins_negative_reviews[(i*100):(i+1)*100] for i in range(math.ceil(len(origins_negative_reviews)/100))]

    for i in range(len(origins_negative_nlp)):
        try:
            origins_negative_nlp_aggregate += generate_summary(origins_negative_nlp[i])
        except:
            continue
        
    return origins_negative_nlp_aggregate

def displayOriginsNegativeNLP(origins_negative_nlp_aggregate):
    origins_negative_nlp_aggregate_final = generate_summary(origins_negative_nlp_aggregate)

    for i in range(len(origins_negative_nlp_aggregate_final)):
        print(f'{origins_negative_nlp_aggregate_final[i]}')
        print('---')

def main():
    origins_negative_reviews = []
    origins_negative_nlp_aggregate = []
    filtered_review_list = reviewsFilterByLength(origins_negative_reviews)
    filtered_review_list_nlp = reviewsNLPByBlock(origins_negative_nlp_aggregate, origins_negative_reviews)
    displayOriginsNegativeNLP(origins_negative_nlp_aggregate)

if __name__ == "__main__":
    main()

Space exploration game that makes you spend 8-15 hours on base building before you can upgrade and explore. What a waste, been waiting nine years to play some cohesive and this is what they give us.  All the free updates in the world could not fix this train wreck of a game. UNINSTALLED>
---
I like the game but Steam will only give me a binary choice in rating it.

Milestones are very annoying and interruptive because they hide the interface. I just don't care if I've done X thing 10 or 20 times, or traveled X km. There should be a way to disable this in the settings.

The game also has a major bug with saving progress. I just lost 2 hours of progress (in which I had been in and out of my ship multiple times).
---
The game is, to quote what others have already said, "as wide as a sea but deep as a puddle". The single player campaign is uninspiring and told mainly through text. Missions are monotonous and repetitive, and give very little sense of accomplishment or satisfaction. What the

#### Processing: Next Generation

In [51]:
def reviewsFilterByLength(next_gen_positive_reviews):
    for i in range(len(nms_next_generation['Reviews'].loc[(nms_next_generation['Recommended?'] == True)].values)):
        next_gen_positive_reviews.append(nms_next_generation['Reviews'].loc[(nms_next_generation['Recommended?'] == True)].values[i])

    return next_gen_positive_reviews

def reviewsNLPByBlock(next_gen_positive_nlp_aggregate, next_gen_positive_reviews):
    next_gen_positive_nlp = [next_gen_positive_reviews[(i*100):(i+1)*100] for i in range(math.ceil(len(next_gen_positive_reviews)/100))]

    for i in range(len(next_gen_positive_nlp)):
        try:
            next_gen_positive_nlp_aggregate += generate_summary(next_gen_positive_nlp[i])
        except:
            continue
    
    return next_gen_positive_nlp_aggregate

def displaNextGenPositiveNLP(next_gen_positive_nlp_aggregate):
    next_gen_positive_nlp_aggregate_final = generate_summary(next_gen_positive_nlp_aggregate)

    for i in range(len(next_gen_positive_nlp_aggregate_final)):
        print(f'{next_gen_positive_nlp_aggregate_final[i]}')
        print('---')

def main():
    next_gen_positive_reviews = []
    next_gen_positive_nlp_aggregate = []
    filtered_review_list = reviewsFilterByLength(next_gen_positive_reviews)
    filtered_review_list_nlp = reviewsNLPByBlock(next_gen_positive_nlp_aggregate, next_gen_positive_reviews)
    displaNextGenPositiveNLP(next_gen_positive_nlp_aggregate)

if __name__ == "__main__":
    main()

Sure No mans sky had a rocky beginning, but haven't most games? The Devs totally rocked this game out of its world with constant updates and patches giving us new things to do and new places to go. I think most players have barely scratched the surface of the game while playing hundred of hours. Maybe this game isn't for everyone but that's fine, its story rich quests and dare i say large open world totally makes anyone feel like a real life astronaut. Thanks, Hello Games.
---
This game serriously is the deffinition of the labour of love. It started out as an over hyped mess brought on by outside factors and inexperience in PR. Really if you havent seen the engoodening of no mans sky, you should watch the video. It really shows that the devs really care about this game and wish to continue to update and fix it. Overrall i give it a 7/10 its good, and if you like space exploration then i would reccomend the game to you
---
This game has to be the biggest labor of love considering it's l

In [52]:
def reviewsFilterByLength(next_gen_negative_reviews):
    for i in range(len(nms_next_generation['Reviews'].loc[(nms_next_generation['Recommended?'] == False)].values)):
        next_gen_negative_reviews.append(nms_next_generation['Reviews'].loc[(nms_next_generation['Recommended?'] == False)].values[i])
    
    return next_gen_negative_reviews

def reviewsNLPByBlock(next_gen_negative_nlp_aggregate, next_gen_negative_reviews):
    next_gen_negative_nlp = [next_gen_negative_reviews[(i*100):(i+1)*100] for i in range(math.ceil(len(next_gen_negative_reviews)/100))]

    for i in range(len(next_gen_negative_nlp)):
        try:
            next_gen_negative_nlp_aggregate += generate_summary(next_gen_negative_nlp[i])
        except:
            continue
        
    return next_gen_negative_nlp_aggregate

def displayNextGenNegativeNLP(next_gen_negative_nlp_aggregate):
    next_gen_negative_nlp_aggregate_final = generate_summary(next_gen_negative_nlp_aggregate)

    for i in range(len(next_gen_negative_nlp_aggregate_final)):
        print(f'{next_gen_negative_nlp_aggregate_final[i]}')
        print('---')

def main():
    next_gen_negative_reviews = []
    next_gen_negative_nlp_aggregate = []
    filtered_review_list = reviewsFilterByLength(next_gen_negative_reviews)
    filtered_review_list_nlp = reviewsNLPByBlock(next_gen_negative_nlp_aggregate, next_gen_negative_reviews)
    displayNextGenNegativeNLP(next_gen_negative_nlp_aggregate)

if __name__ == "__main__":
    main()

What a dumpster fire of a game. The inventory system & management makes no sense. I lost 3 hours of game play due to the game crashing after I died. The options menu only has arrow toggle to change certain options. No slide bar or the ability to input values. The build menu is so Janky & awkward to navigate. I feel like the system was made by a pseudo masochist that enjoys watching people suffer. God help humanity!
---
When having too much random freedom is undesirable. I love flying around the galaxies scanning anything and everything like I was a planetary cashier but honestly it gets boring after a while. Give me more meaningful co-op missions, more ways to earn quicksilver for those cosmetic upgrades, more input as to how "far" I am in the game. It's worth a try to kill some time but it'll leave you wanting.
---
Visually crude, the VR version is a generation behind other recent games like Half-Life Alyx.  The gameplay is as much fun as going to work and putting a new cover on the T

#### Processing: Companions

In [53]:
def reviewsFilterByLength(companions_positive_reviews):
    for i in range(len(nms_companions['Reviews'].loc[(nms_companions['Recommended?'] == True)].values)):
        companions_positive_reviews.append(nms_companions['Reviews'].loc[(nms_companions['Recommended?'] == True)].values[i])

    return companions_positive_reviews

def reviewsNLPByBlock(companions_positive_nlp_aggregate, companions_positive_reviews):
    companions_positive_nlp = [companions_positive_reviews[(i*100):(i+1)*100] for i in range(math.ceil(len(companions_positive_reviews)/100))]

    for i in range(len(companions_positive_nlp)):
        try:
            companions_positive_nlp_aggregate += generate_summary(companions_positive_nlp[i])
        except:
            continue
    
    return companions_positive_nlp_aggregate

def displayCompanionsPositiveNLP(companions_positive_nlp_aggregate):
    companions_positive_nlp_aggregate_final = generate_summary(companions_positive_nlp_aggregate)

    for i in range(len(companions_positive_nlp_aggregate_final)):
        print(f'{companions_positive_nlp_aggregate_final[i]}')
        print('---')

def main():
    companions_positive_reviews = []
    companions_positive_nlp_aggregate = []
    filtered_review_list = reviewsFilterByLength(companions_positive_reviews)
    filtered_review_list_nlp = reviewsNLPByBlock(companions_positive_nlp_aggregate, companions_positive_reviews)
    displayCompanionsPositiveNLP(companions_positive_nlp_aggregate)

if __name__ == "__main__":
    main()

It's just a really enjoyable game if you want to waste time and be engaged. I just recently started playing in VR and holy crap it's a whole new game. The graphics are kind of outdated, but the game is so rich in play and exploration that the graphics dont even become a realization.
---
This game is much closer to what was originally promised, it's honestly the best survival/exploration game out there. It still gets free updates frequently so you wont get bored. Right now the game has around 200 hours of content and with multiplayer ontop of that, you wont run out of things to do.

The devs are clearly passionate about this game and want to make an enjoyable experience and I think after 3 years of updates they have now gone beyond that.

9/10 :)
---
If you go into this game expecting to fly around space and enjoy looking at weird and beautiful planets and creatures, you won't be disappointed. If you're expecting anything more than that I think you will be. 

I've played this game for ~

In [54]:
def reviewsFilterByLength(companions_negative_reviews):
    for i in range(len(nms_companions['Reviews'].loc[(nms_companions['Recommended?'] == False)].values)):
        companions_negative_reviews.append(nms_companions['Reviews'].loc[(nms_companions['Recommended?'] == False)].values[i])
    
    return companions_negative_reviews

def reviewsNLPByBlock(companions_negative_nlp_aggregate, companions_negative_reviews):
    companions_negative_nlp = [companions_negative_reviews[(i*100):(i+1)*100] for i in range(math.ceil(len(companions_negative_reviews)/100))]

    for i in range(len(companions_negative_nlp)):
        try:
            companions_negative_nlp_aggregate += generate_summary(companions_negative_nlp[i])
        except:
            continue
        
    return companions_negative_nlp_aggregate

def displayCompanionsNegativeNLP(companions_negative_nlp_aggregate):
    companions_negative_nlp_aggregate_final = generate_summary(companions_negative_nlp_aggregate)

    for i in range(len(companions_negative_nlp_aggregate_final)):
        print(f'{companions_negative_nlp_aggregate_final[i]}')
        print('---')

def main():
    companions_negative_reviews = []
    companions_negative_nlp_aggregate = []
    filtered_review_list = reviewsFilterByLength(companions_negative_reviews)
    filtered_review_list_nlp = reviewsNLPByBlock(companions_negative_nlp_aggregate, companions_negative_reviews)
    displayCompanionsNegativeNLP(companions_negative_nlp_aggregate)

if __name__ == "__main__":
    main()

I really want to like this game, and it's got potential to be great, but man have I experienced some bugs; from the ground on planets not loading to the game crashing I have experienced a plethora of both game-breaking and just visual bugs. I would be fine if the game was marked as early access. but as a full supposedly finished game I can not justify the 30 odd dollars I spent on it. I can not recommend anyone buying this game for more then 15 dollars.
---
I tried the single player and multiplayer with friends over the last few days. Unfortunately the game is an incomprehensible mess. It doesn't explain itself at all, and doesn't have good enough path finding for you to work it out yourself. Other similar games do a much better job easing in to the world and mechanics. The constant beeping from suit warnings will make you want to put a hammer through your monitor.

This is the most disappointing game I've played in years.
---
The general consensus seems to be that it offers about 60-8

#### Processing: Expeditions

In [55]:
def reviewsFilterByLength(expeditions_positive_reviews):
    for i in range(len(nms_expeditions['Reviews'].loc[(nms_expeditions['Recommended?'] == True)].values)):
        expeditions_positive_reviews.append(nms_expeditions['Reviews'].loc[(nms_expeditions['Recommended?'] == True)].values[i])

    return expeditions_positive_reviews

def reviewsNLPByBlock(expeditions_positive_nlp_aggregate, expeditions_positive_reviews):
    expeditions_positive_nlp = [expeditions_positive_reviews[(i*100):(i+1)*100] for i in range(math.ceil(len(expeditions_positive_reviews)/100))]

    for i in range(len(expeditions_positive_nlp)):
        try:
            expeditions_positive_nlp_aggregate += generate_summary(expeditions_positive_nlp[i])
        except:
            continue
    
    return expeditions_positive_nlp_aggregate

def displayExpeditionsPositiveNLP(expeditions_positive_nlp_aggregate):
    expeditions_positive_nlp_aggregate_final = generate_summary(expeditions_positive_nlp_aggregate)

    for i in range(len(expeditions_positive_nlp_aggregate_final)):
        print(f'{expeditions_positive_nlp_aggregate_final[i]}')
        print('---')

def main():
    expeditions_positive_reviews = []
    expeditions_positive_nlp_aggregate = []
    filtered_review_list = reviewsFilterByLength(expeditions_positive_reviews)
    filtered_review_list_nlp = reviewsNLPByBlock(expeditions_positive_nlp_aggregate, expeditions_positive_reviews)
    displayExpeditionsPositiveNLP(expeditions_positive_nlp_aggregate)

if __name__ == "__main__":
    main()

I had a bad review on this game for a while but they definitely picked this game back up again. It's honestly a really fun immersive experience if you're willing to put the time into it and give it a chance. All the updates have really turned this game from an unenjoyable mess to a fun time if you like grinding and exploring at the same time. 

---
I Bought this game when the foundation patch was released, used mods to improve scenery etc.  Mods are no longer needed as this game looks and sounds incredible now.  Never seen a game expand and improve so much in such a short space of time.  The tasks seem to be never ending, so will keep you very busy, great work Hello Games for not giving up on this and the Expedition mode is a brilliant idea.  Planet i'm on is just riddled with players bases.
---
This game has had a wild history but is in a really good place now. Hello Games have spent the last few years adding tons of content to this game with no signs of stopping, and there is frankly

In [56]:
def reviewsFilterByLength(expeditions_negative_reviews):
    for i in range(len(nms_expeditions['Reviews'].loc[(nms_expeditions['Recommended?'] == False)].values)):
        expeditions_negative_reviews.append(nms_expeditions['Reviews'].loc[(nms_expeditions['Recommended?'] == False)].values[i])
    
    return expeditions_negative_reviews

def reviewsNLPByBlock(expeditions_negative_nlp_aggregate, expeditions_negative_reviews):
    expeditions_negative_nlp = [expeditions_negative_reviews[(i*100):(i+1)*100] for i in range(math.ceil(len(expeditions_negative_reviews)/100))]

    for i in range(len(expeditions_negative_nlp)):
        try:
            expeditions_negative_nlp_aggregate += generate_summary(expeditions_negative_nlp[i])
        except:
            continue
        
    return expeditions_negative_nlp_aggregate

def displayExpeditionsNegativeNLP(expeditions_negative_nlp_aggregate):
    expeditions_negative_nlp_aggregate_final = generate_summary(expeditions_negative_nlp_aggregate)

    for i in range(len(expeditions_negative_nlp_aggregate_final)):
        print(f'{expeditions_negative_nlp_aggregate_final[i]}')
        print('---')

def main():
    expeditions_negative_reviews = []
    expeditions_negative_nlp_aggregate = []
    filtered_review_list = reviewsFilterByLength(expeditions_negative_reviews)
    filtered_review_list_nlp = reviewsNLPByBlock(expeditions_negative_nlp_aggregate, expeditions_negative_reviews)
    displayExpeditionsNegativeNLP(expeditions_negative_nlp_aggregate)

if __name__ == "__main__":
    main()

I wanted to like this game, but I just couldn't. For a game with seemingly infinite potential for exploration, it feels so limited and empty. The story was boring. They planets, despite having various differences, they all feel the same. All you do is gather materials and build stuff. Ships aren't aesthetically customizable. I can appreciate the fact that the Devs work on this continuously, but it probably needs another 4 years of support to make this game enjoyable for me.
---
it has been nearly 6 years for you guys to get your shit together and make something in which my friends and i can collaborate together to complete missions as a party; yet, you guys have somehow dropped the ball an innumerable amount of times. im done. suck my balls and i hope you guys used the money I paid for this game 5 years ago to go back to multiplayer-development school
---
Don't look at the time played, I played much more hours. I came back many times on this game, and everytime I figure out how boring 

#### Processing: Prisms

In [57]:
def reviewsFilterByLength(prisms_positive_reviews):
    for i in range(len(nms_prisms['Reviews'].loc[(nms_prisms['Recommended?'] == True)].values)):
        prisms_positive_reviews.append(nms_prisms['Reviews'].loc[(nms_prisms['Recommended?'] == True)].values[i])

    return prisms_positive_reviews

def reviewsNLPByBlock(prisms_positive_nlp_aggregate, prisms_positive_reviews):
    prisms_positive_nlp = [prisms_positive_reviews[(i*100):(i+1)*100] for i in range(math.ceil(len(prisms_positive_reviews)/100))]

    for i in range(len(prisms_positive_nlp)):
        try:
            prisms_positive_nlp_aggregate += generate_summary(prisms_positive_nlp[i])
        except:
            continue
    
    return prisms_positive_nlp_aggregate

def displayPrismsPositiveNLP(prisms_positive_nlp_aggregate):
    prisms_positive_nlp_aggregate_final = generate_summary(prisms_positive_nlp_aggregate)

    for i in range(len(prisms_positive_nlp_aggregate_final)):
        print(f'{prisms_positive_nlp_aggregate_final[i]}')
        print('---')

def main():
    prisms_positive_reviews = []
    prisms_positive_nlp_aggregate = []
    filtered_review_list = reviewsFilterByLength(prisms_positive_reviews)
    filtered_review_list_nlp = reviewsNLPByBlock(prisms_positive_nlp_aggregate, prisms_positive_reviews)
    displayPrismsPositiveNLP(prisms_positive_nlp_aggregate)

if __name__ == "__main__":
    main()

One of the very best open world games. Confusing at times during startup (where is that hermetic seal?), but it is a game after all. And the developer is in for the long haul - not many games can say that.
---
The games is a lot of fun, and hello games have really picked this up and made this game truly worth it. A rare comeback story of a game developers deciding to take their broken game and made is amazing. Its a lot of fun and is easy to just relax and explore or just go out and do your own thing, maybe one of the only problems are minimal bugs, but with consistent developer support and mainline updates, that is more then fine. Consider getting the game! its worth it! Have fun.
---
If you are tempted to buy this game do it. It has some good mechanics some recent major updates and a variety of things to do farming, trading and specially EXPLORING you'll do a lot of a that and its really fun. I enjoy type games like these which gives some sort of freedom when exploring such as new ga

In [58]:
def reviewsFilterByLength(prisms_negative_reviews):
    for i in range(len(nms_prisms['Reviews'].loc[(nms_prisms['Recommended?'] == False)].values)):
        prisms_negative_reviews.append(nms_prisms['Reviews'].loc[(nms_prisms['Recommended?'] == False)].values[i])
    
    return prisms_negative_reviews

def reviewsNLPByBlock(prisms_negative_nlp_aggregate, prisms_negative_reviews):
    prisms_negative_nlp = [prisms_negative_reviews[(i*100):(i+1)*100] for i in range(math.ceil(len(prisms_negative_reviews)/100))]

    for i in range(len(prisms_negative_nlp)):
        try:
            prisms_negative_nlp_aggregate += generate_summary(prisms_negative_nlp[i])
        except:
            continue
        
    return prisms_negative_nlp_aggregate

def displayPrismsNegativeNLP(prisms_negative_nlp_aggregate):
    prisms_negative_nlp_aggregate_final = generate_summary(prisms_negative_nlp_aggregate)

    for i in range(len(prisms_negative_nlp_aggregate_final)):
        print(f'{prisms_negative_nlp_aggregate_final[i]}')
        print('---')

def main():
    prisms_negative_reviews = []
    prisms_negative_nlp_aggregate = []
    filtered_review_list = reviewsFilterByLength(prisms_negative_reviews)
    filtered_review_list_nlp = reviewsNLPByBlock(prisms_negative_nlp_aggregate, prisms_negative_reviews)
    displayPrismsNegativeNLP(prisms_negative_nlp_aggregate)

if __name__ == "__main__":
    main()

Border line playable GUI for PC mouse users, countless minutes if not hours wasted on press and hold click to make a selection, even more wasted time on all the text animation and and the motion sickness inducing camera pans. on the 2K wide screen mode you'r so close to your toon the it's ass covers 30% of the view, not that kind of a play i'm looking for.
I really want to play this game and i keep trying but GUI is always whats pushes me over the edge to quit
---
For a 5 year old game, lots of basic steam achievements have been claimed by owners. Its a grind, and not in a fun way. At high levels you need hundred of thousands for nanites. You can earn 6-8K per hour by holding E then click threw dialog, repeat. 
I'd talk about more annoyances but doubt there is enough room
---
game has tipped in the balance of the monsters.  also frustrating that you can't play due to universe not loading 95% of the time.  can't enjoy a game you bought if you can't play it.  needs alot more work to make

After each patch's reviews are broken down into positive and negative categories, the summarisation algorithm stack ranks reviews from each category and determines the top five reviews that best represents the patch both positively and negatively.

All patches, regardless of size, are initially divided into groups of 100. After finding the top five reviews from each group of 100, they are then added into another graph to repeat the process again. A bit of error handling was also used just in case if the algorithm had a really bad batch of reviews and returned something that might cause issues. It should not given how things were set up, but once again just in case...

And that should do it! Both the numbers from the analytics notebook and the reviews from this notebook should be enough for the Tableau storyboard.

## Credits
Immense gratitude and special thanks to <b>[Forerunners](https://www.youtube.com/channel/UCOv8xYSnv27f-z5SiKLYajQ)</b> for their [code](https://www.youtube.com/watch?v=dFe7tbH39Eg), which made everything possible. While the <i>NL toolkit</i> library is well-known to many, the <i>NetworkX</i> library is not. Its function, [pagerank](https://networkx.org/documentation/stable/reference/algorithms/generated/networkx.algorithms.link_analysis.pagerank_alg.pagerank.html), is the key to making summarisation possible by ranking review nodes in relative to each other. This project could have never been realised without such a library like <i>NetworkX</i>, and any continuation in this topic would have been all the more difficult without knowledge of its existence.

Also once again many thanks to <b>Hello Games</b> for giving the world <b>No Man's Sky</b> as well.